# LeetCode 126 — Word Ladder II (Hard): nhập môn Đồ thị, BFS theo tầng và Backtracking bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findLadders(self, beginWord: str, endWord: str, wordList: List[str]) -> List[List[str]]:

```

Bài này nhìn thì giống trò chơi chữ, nhưng thực chất là bài **tìm TẤT CẢ các đường đi ngắn nhất trong một đồ thị**. Nó là bài "anh lớn" của #127 Word Ladder (bài #127 chỉ hỏi *độ dài* đường ngắn nhất; bài này đòi *liệt kê mọi* đường ngắn nhất). Notebook này **tự đủ**: bạn không cần học #127 trước.

---

## Sau notebook này, bạn sẽ nắm được

1. Đề bài yêu cầu gì, kiểm tra một "chuỗi biến đổi" hợp lệ bằng tay và bằng code.
2. **Đồ thị (graph)** là gì: đỉnh, cạnh, danh sách kề (adjacency list) — và vì sao mỗi từ là một đỉnh.
3. **Đệ quy, DFS và backtracking** (chọn → đi tiếp → bỏ chọn) — dùng để liệt kê đường đi (Cách 1, rất chậm, chỉ để hiểu).
4. **Hàng đợi `deque` và BFS** (loang theo từng vòng) — vì sao BFS cho **đường ngắn nhất**.
5. Cách 2: BFS lưu nguyên đường đi trong hàng đợi — đúng nhưng tốn bộ nhớ.
6. **Insight chính**: chỉ cần lưu **đồ thị cha** `parents` (DAG theo tầng), rồi **backtrack từ `endWord` ngược về `beginWord`**; vì sao phải **xóa từ theo TẦNG** chứ không xóa ngay khi gặp.
7. Ba cách sinh "hàng xóm": so từng cặp, thay từng ký tự `a–z`, và **wildcard bucket** `"h*t"`.
8. (Tùy chọn) **BFS hai chiều** (bidirectional BFS).
9. Lỗi hay gặp, mẹo nhận dạng pattern, biến thể và 6 bài tập có đáp án ở cuối.

## 1. Đề bài

Một **chuỗi biến đổi** (transformation sequence) từ từ `beginWord` tới từ `endWord` dùng từ điển `wordList` là một dãy từ `beginWord -> s1 -> s2 -> ... -> sk` sao cho:

- Mỗi cặp từ **liền kề** trong dãy khác nhau **đúng một chữ cái**.
- Mọi `si` với `1 <= i <= k` đều nằm trong `wordList`. Lưu ý: `beginWord` **không cần** nằm trong `wordList`.
- `sk == endWord`.

Cho hai từ `beginWord`, `endWord` và từ điển `wordList`, hãy trả về **tất cả các chuỗi biến đổi ngắn nhất** từ `beginWord` tới `endWord`, hoặc danh sách rỗng nếu không có chuỗi nào. Mỗi chuỗi được trả về dưới dạng danh sách các từ `[beginWord, s1, s2, ..., sk]`.

**Example 1**

```text
Input: beginWord = "hit", endWord = "cog", wordList = ["hot","dot","dog","lot","log","cog"]
Output: [["hit","hot","dot","dog","cog"],["hit","hot","lot","log","cog"]]
Explanation: There are 2 shortest transformation sequences:
"hit" -> "hot" -> "dot" -> "dog" -> "cog"
"hit" -> "hot" -> "lot" -> "log" -> "cog"
```

**Example 2**

```text
Input: beginWord = "hit", endWord = "cog", wordList = ["hot","dot","dog","lot","log"]
Output: []
Explanation: The endWord "cog" is not in wordList, therefore there is no valid transformation sequence.
```

**Constraints**

- `1 <= beginWord.length <= 5`
- `endWord.length == beginWord.length`
- `1 <= wordList.length <= 500`
- `wordList[i].length == beginWord.length`
- `beginWord`, `endWord`, và `wordList[i]` chỉ gồm chữ cái tiếng Anh viết thường.
- `beginWord != endWord`
- Tất cả các từ trong `wordList` là **khác nhau** (unique).
- Tổng (kích thước) của tất cả các chuỗi biến đổi ngắn nhất không vượt quá `10^5`.

Các chuỗi kết quả có thể trả về theo **thứ tự bất kỳ** — vì vậy trong notebook, khi so sánh kết quả ta luôn `sorted(...)` trước.

In [1]:
from typing import List
from collections import defaultdict, deque
import random
import itertools

EXAMPLE_1 = ("hit", "cog", ["hot", "dot", "dog", "lot", "log", "cog"])
EXAMPLE_2 = ("hit", "cog", ["hot", "dot", "dog", "lot", "log"])
EXPECTED_1 = [["hit", "hot", "dot", "dog", "cog"], ["hit", "hot", "lot", "log", "cog"]]
EXPECTED_2 = []

def show_paths(paths):
    # In từng đường trên một dòng, sắp xếp để output luôn ổn định
    if not paths:
        print("  (không có đường nào)")
    for p in sorted(paths):
        print("  " + " -> ".join(p))

begin_word, end_word, word_list = EXAMPLE_1
print("beginWord =", begin_word, "| endWord =", end_word)
print("wordList  =", word_list)
print("Kết quả mong đợi:")
show_paths(EXPECTED_1)

beginWord = hit | endWord = cog
wordList  = ['hot', 'dot', 'dog', 'lot', 'log', 'cog']
Kết quả mong đợi:
  hit -> hot -> dot -> dog -> cog
  hit -> hot -> lot -> log -> cog


## 2. Hiểu đề bằng tay

Ta hãy "chơi" Example 1 bằng tay. Từ `hit`, đổi **một** chữ để ra một từ có trong `wordList`:

- `hit` → đổi `i` thành `o` → `hot` ✅ (có trong `wordList`)
- `hit` → `hat`, `hip`, `bit`... ❌ (không có trong `wordList`)

Tiếp tục:

| Bước | Từ hiện tại | Đổi chữ nào | Từ mới | Hợp lệ? |
|---:|---|---|---|---|
| 1 | `hit` | `i → o` (vị trí 1) | `hot` | ✅ |
| 2 | `hot` | `h → d` (vị trí 0) | `dot` | ✅ |
| 3 | `dot` | `t → g` (vị trí 2) | `dog` | ✅ |
| 4 | `dog` | `d → c` (vị trí 0) | `cog` | ✅ = `endWord` |

Ta được một chuỗi dài **5 từ** (4 lần biến đổi). Còn có một chuỗi khác cũng dài 5 từ: `hit -> hot -> lot -> log -> cog`.

Có chuỗi **dài hơn** không? Có! Ví dụ `hit -> hot -> dot -> lot -> log -> cog` (6 từ) — hợp lệ nhưng **không ngắn nhất**, nên **không** được đưa vào đáp án.

Ba điều cần khắc ghi:

1. Đề muốn **mọi** chuỗi **ngắn nhất** — không phải một chuỗi, cũng không phải mọi chuỗi.
2. `beginWord` không cần có trong `wordList`, nhưng **`endWord` bắt buộc phải có** (Example 2 trả `[]` vì thiếu `cog`).
3. Độ dài được đo bằng **số từ** trong chuỗi (hoặc tương đương: số lần biến đổi + 1).

## 3. Nền tảng 1 — Hai từ "kề nhau": khác đúng một ký tự

Thao tác cơ bản nhất của bài: kiểm tra hai từ cùng độ dài có khác nhau **đúng 1 vị trí** không.

Ta dùng `zip(a, b)`: nó ghép từng cặp ký tự cùng vị trí.

```python
list(zip("hit", "hot"))   # [('h','h'), ('i','o'), ('t','t')]
```

Đếm số cặp khác nhau: `sum(x != y for x, y in zip(a, b))`. Trong Python, `True` được tính là `1`, `False` là `0`, nên `sum` của một dãy True/False chính là **số lần True**.

- `"hit"` vs `"hot"` → khác 1 vị trí → **kề nhau**.
- `"hit"` vs `"cog"` → khác 3 vị trí → không kề.
- `"dot"` vs `"dot"` → khác 0 vị trí → không kề (một từ không "kề" chính nó).

In [2]:
print(list(zip("hit", "hot")))
print([x != y for x, y in zip("hit", "hot")])

def differ_by_one(a: str, b: str) -> bool:
    # Đếm số vị trí khác nhau; kề nhau <=> đúng 1 vị trí
    return sum(x != y for x, y in zip(a, b)) == 1

for a, b in [("hit", "hot"), ("hit", "cog"), ("dot", "dot"), ("dot", "lot")]:
    print(f"{a} vs {b}: differ_by_one = {differ_by_one(a, b)}")

[('h', 'h'), ('i', 'o'), ('t', 't')]
[False, True, False]
hit vs hot: differ_by_one = True
hit vs cog: differ_by_one = False
dot vs dot: differ_by_one = False
dot vs lot: differ_by_one = True


### Viết hàm kiểm tra một chuỗi biến đổi có hợp lệ không

Trước khi giải, ta tự viết "trọng tài" `is_valid_ladder` để sau này kiểm tra mọi đáp án. Nó kiểm đúng 3 điều kiện của đề:

1. Từ đầu là `beginWord`, từ cuối là `endWord`.
2. Mọi từ **sau** từ đầu đều thuộc `wordList`.
3. Mọi cặp liền kề khác đúng 1 ký tự.

`zip(seq, seq[1:])` là mẹo duyệt **từng cặp liền kề**: với `seq = [a, b, c]` nó cho `(a, b), (b, c)`.

In [3]:
def is_valid_ladder(begin_word, end_word, word_list, seq):
    word_set = set(word_list)
    if not seq or seq[0] != begin_word or seq[-1] != end_word:
        return False
    if any(w not in word_set for w in seq[1:]):
        return False
    return all(differ_by_one(a, b) for a, b in zip(seq, seq[1:]))

print(list(zip(["a", "b", "c"], ["a", "b", "c"][1:])))
print(is_valid_ladder(*EXAMPLE_1, ["hit", "hot", "dot", "dog", "cog"]))           # True
print(is_valid_ladder(*EXAMPLE_1, ["hit", "hot", "dot", "lot", "log", "cog"]))    # True nhưng KHÔNG ngắn nhất
print(is_valid_ladder(*EXAMPLE_1, ["hit", "dot", "dog", "cog"]))                 # False: hit -> dot khác 2 chữ
print(is_valid_ladder(*EXAMPLE_2, ["hit", "hot", "dot", "dog", "cog"]))           # False: cog không có trong wordList

[('a', 'b'), ('b', 'c')]
True
True
False
False


## 4. Nền tảng 2 — Đồ thị (graph)

**Đồ thị** (graph) gồm:

- **Đỉnh** (vertex / node): các "điểm". Ở bài này: **mỗi từ là một đỉnh** (gồm `beginWord` và mọi từ trong `wordList`).
- **Cạnh** (edge): nối hai đỉnh có quan hệ. Ở bài này: hai từ **khác đúng 1 chữ** thì có cạnh. Cạnh **vô hướng** (undirected): `hot – dot` thì đi được cả hai chiều.

Đồ thị của Example 1 trông như sau:

```text
              dot ─────── dog
             /  |          |  \
 hit ──── hot   |          |   cog
             \  |          |  /
              lot ─────── log
```

(`dot – lot` có cạnh vì khác đúng chữ đầu; `dog – log` cũng vậy.)

Khi đó, "chuỗi biến đổi" chính là một **đường đi** (path) trên đồ thị từ đỉnh `hit` tới đỉnh `cog`, và "ngắn nhất" là đường đi qua **ít cạnh nhất**.

👉 Bài toán trở thành: **liệt kê mọi đường đi ngắn nhất từ `beginWord` tới `endWord` trong đồ thị**.

### Lưu đồ thị bằng danh sách kề (adjacency list)

Cách lưu phổ biến nhất: một `dict` mà **key là đỉnh**, **value là list các đỉnh kề**:

```python
graph = {"hit": ["hot"], "hot": ["hit", "dot", "lot"], ...}
```

Cách dựng "ngây thơ" đầu tiên mà ai cũng nghĩ ra: **so từng cặp từ** bằng `differ_by_one`.

In [4]:
def build_graph_pairwise(begin_word, word_list):
    # Đỉnh = beginWord + các từ trong wordList (bỏ trùng nếu beginWord đã có trong list)
    nodes = [begin_word] + [w for w in word_list if w != begin_word]
    graph = {w: [] for w in nodes}
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            if differ_by_one(nodes[i], nodes[j]):
                graph[nodes[i]].append(nodes[j])     # cạnh vô hướng: thêm cả 2 chiều
                graph[nodes[j]].append(nodes[i])
    return graph

graph_1 = build_graph_pairwise(EXAMPLE_1[0], EXAMPLE_1[2])
for word, nbs in graph_1.items():
    print(f"{word}: {nbs}")

hit: ['hot']
hot: ['hit', 'dot', 'lot']
dot: ['hot', 'dog', 'lot']
dog: ['dot', 'log', 'cog']
lot: ['hot', 'dot', 'log']
log: ['dog', 'lot', 'cog']
cog: ['dog', 'log']


Đọc output: `hot` kề với `hit`, `dot`, `lot` — đúng như hình vẽ. Mỗi cạnh xuất hiện **2 lần** (ở cả hai đầu), ví dụ `hot` có `dot` và `dot` có `hot`.

Chi phí dựng bằng cách so từng cặp: có khoảng `N²/2` cặp, mỗi cặp so `L` ký tự → `O(N² · L)` với `N` = số từ, `L` = độ dài từ. Với `N = 500`, `L = 5` là khoảng 625 nghìn phép so ký tự — chạy được, nhưng ở mục 12 ta sẽ thấy cách nhanh hơn nhiều.

## 5. Nền tảng 3 — `set`, `dict` và `defaultdict(list)`

Ba công cụ sẽ dùng liên tục:

| Công cụ | Dùng để | Thao tác chính | Tốc độ |
|---|---|---|---|
| `set` | tập "đã thăm / chưa thăm" | `add`, `discard`, `in`, `-=` (bớt cả tập) | `in` trung bình `O(1)` |
| `dict` | ánh xạ từ → thông tin (khoảng cách, cha...) | `d[k] = v`, `d[k]`, `k in d` | trung bình `O(1)` |
| `defaultdict(list)` | dict mà key chưa có thì **tự tạo list rỗng** | `d[k].append(x)` | như dict |

Vì sao cần `defaultdict(list)`? Với `dict` thường, `d["cog"].append("dog")` sẽ báo `KeyError` nếu `"cog"` chưa có. `defaultdict(list)` tự tạo `[]` cho key mới, rất hợp để gom "danh sách cha" cho mỗi từ.

Lưu ý quan trọng: **thứ tự duyệt một `set` chứa chuỗi có thể khác nhau giữa các lần chạy** (Python xáo hash của chuỗi mỗi lần khởi động). Vì thế khi *in* một set, ta luôn `sorted(...)` để output ổn định.

In [5]:
unvisited = {"hot", "dot", "dog"}
print("dot" in unvisited, "cat" in unvisited)
unvisited -= {"dot", "dog"}          # bớt nhiều phần tử một lúc
print(sorted(unvisited))

parents = defaultdict(list)
parents["cog"].append("dog")         # key "cog" chưa có -> tự tạo [] rồi append
parents["cog"].append("log")
parents["dog"].append("dot")
print(dict(parents))

try:
    normal = {}
    normal["cog"].append("dog")
except KeyError as e:
    print("dict thường báo KeyError:", e)

True False
['hot']
{'cog': ['dog', 'log'], 'dog': ['dot']}
dict thường báo KeyError: 'cog'


## 6. Nền tảng 4 — Đệ quy, DFS và backtracking

### 6.1 Đệ quy (recursion)

**Đệ quy** là khi một hàm **gọi lại chính nó** với bài toán nhỏ hơn. Mọi hàm đệ quy cần:

1. **Điểm dừng** (base case): trường hợp đủ nhỏ, trả lời ngay.
2. **Bước đệ quy**: gọi chính nó với input "gần điểm dừng hơn".

Mỗi lần gọi hàm, Python đặt một "khung" lên **call stack** (ngăn xếp lời gọi); hàm con chạy xong thì khung bị gỡ và quay về hàm cha. Ví dụ đếm ngược:

In [6]:
def countdown(n, depth=0):
    indent = "  " * depth
    if n == 0:                       # điểm dừng
        print(indent + "chạm đáy: n = 0")
        return
    print(indent + f"vào countdown({n})")
    countdown(n - 1, depth + 1)      # bước đệ quy: bài nhỏ hơn
    print(indent + f"quay về countdown({n})")

countdown(3)

vào countdown(3)
  vào countdown(2)
    vào countdown(1)
      chạm đáy: n = 0
    quay về countdown(1)
  quay về countdown(2)
quay về countdown(3)


### 6.2 DFS và khuôn mẫu backtracking: chọn → đi tiếp → bỏ chọn

**DFS** (Depth-First Search — tìm kiếm theo chiều sâu): từ một đỉnh, đi **sâu hết mức** theo một nhánh, bế tắc thì **lùi lại** (backtrack) và thử nhánh khác.

Để **liệt kê mọi đường đi**, ta giữ một list `path` là đường đang đi, và theo khuôn mẫu:

```python
path.append(nxt)   # 1. CHỌN: bước sang đỉnh nxt
dfs(nxt)           # 2. ĐI TIẾP
path.pop()         # 3. BỎ CHỌN: lùi về, để thử đỉnh khác
```

Khi chạm đích, ta lưu **bản sao** `path[:]` (giải thích ngay bên dưới vì sao phải sao chép).

Ví dụ mini: đồ thị **có hướng** `A → B, A → C, C → B, B → D, C → D`. Các đường từ `A` tới `D`: `A B D`, `A C B D`, `A C D`.

In [7]:
mini_graph = {"A": ["B", "C"], "B": ["D"], "C": ["B", "D"], "D": []}

def all_paths_mini(graph, start, target):
    result = []
    path = [start]
    def dfs(node):
        if node == target:
            result.append(path[:])       # lưu BẢN SAO của path
            return
        for nxt in graph[node]:
            path.append(nxt)             # chọn
            dfs(nxt)                     # đi tiếp
            path.pop()                   # bỏ chọn
    dfs(start)
    return result

print(all_paths_mini(mini_graph, "A", "D"))

[['A', 'B', 'D'], ['A', 'C', 'B', 'D'], ['A', 'C', 'D']]


### 6.3 Vì sao phải lưu `path[:]` chứ không phải `path`?

`path` là **một** list duy nhất, bị `append`/`pop` liên tục. Nếu ta lưu `result.append(path)`, ta chỉ lưu một **tham chiếu** (reference — "địa chỉ") tới cùng một list. Khi DFS kết thúc, `path` đã bị `pop` về `["A"]`, nên **mọi phần tử** của `result` đều hiện `["A"]`!

`path[:]` (hoặc `list(path)`) tạo một list **mới** chụp lại trạng thái lúc đó.

In [8]:
def all_paths_mini_wrong(graph, start, target):
    result, path = [], [start]
    def dfs(node):
        if node == target:
            result.append(path)          # SAI: lưu tham chiếu
            return
        for nxt in graph[node]:
            path.append(nxt); dfs(nxt); path.pop()
    dfs(start)
    return result

print("Sai :", all_paths_mini_wrong(mini_graph, "A", "D"))
print("Đúng:", all_paths_mini(mini_graph, "A", "D"))

Sai : [['A'], ['A'], ['A']]
Đúng: [['A', 'B', 'D'], ['A', 'C', 'B', 'D'], ['A', 'C', 'D']]


### 6.4 Đồ thị có chu trình: cần nhớ "đang trên đường"

Đồ thị từ của ta là **vô hướng** và có **chu trình** (cycle): `hot → dot → hot → dot → ...` sẽ lặp vô hạn. Vì vậy khi DFS trên đồ thị từ, ta giữ thêm một `set` tên `on_path` chứa các đỉnh **đang nằm trên đường hiện tại**, và **không** bước vào đỉnh đã có trong `on_path`. Khi bỏ chọn, nhớ gỡ đỉnh khỏi `on_path` để nhánh khác còn được dùng nó.

Đây chính là nguyên liệu cho Cách 1.

## 7. Cách 1 — DFS liệt kê MỌI đường đi, rồi lọc đường ngắn nhất

Ý tưởng "người mới tự nghĩ ra":

1. Dựng đồ thị (so từng cặp).
2. DFS từ `beginWord`, liệt kê **mọi** đường đi đơn (không lặp đỉnh) tới `endWord`.
3. Tìm độ dài nhỏ nhất, giữ lại các đường có đúng độ dài đó.

Code này chắc chắn đúng (vì nó thử **tất cả**), nên nó là **"brute force" để đối chiếu** các cách sau trên input nhỏ. Tham số `stats` (tùy chọn) dùng để đếm số đường và số lời gọi `dfs`, giúp ta *thấy* nó tốn kém thế nào.

In [9]:
def find_ladders_dfs_all(begin_word, end_word, word_list, stats=None):
    if end_word not in word_list:
        return []
    graph = build_graph_pairwise(begin_word, word_list)
    all_paths = []
    path = [begin_word]
    on_path = {begin_word}
    calls = 0

    def dfs(word):
        nonlocal calls
        calls += 1
        if word == end_word:
            all_paths.append(path[:])
            return
        for nb in graph[word]:
            if nb in on_path:            # tránh đi vòng
                continue
            on_path.add(nb); path.append(nb)      # chọn
            dfs(nb)                                # đi tiếp
            path.pop(); on_path.remove(nb)        # bỏ chọn

    dfs(begin_word)
    if stats is not None:
        stats["all_paths"] = all_paths
        stats["calls"] = calls
    if not all_paths:
        return []
    best = min(len(p) for p in all_paths)
    return [p for p in all_paths if len(p) == best]

stats = {}
res = find_ladders_dfs_all(*EXAMPLE_1, stats=stats)
print("Mọi đường đơn hit -> cog tìm được:", len(stats["all_paths"]), "| số lời gọi dfs:", stats["calls"])
for p in sorted(stats["all_paths"], key=lambda p: (len(p), p)):
    print(f"  dài {len(p)}: {' -> '.join(p)}")
print("Sau khi lọc ngắn nhất:")
show_paths(res)
print("Example 2:")
show_paths(find_ladders_dfs_all(*EXAMPLE_2))

Mọi đường đơn hit -> cog tìm được: 8 | số lời gọi dfs: 24
  dài 5: hit -> hot -> dot -> dog -> cog
  dài 5: hit -> hot -> lot -> log -> cog
  dài 6: hit -> hot -> dot -> dog -> log -> cog
  dài 6: hit -> hot -> dot -> lot -> log -> cog
  dài 6: hit -> hot -> lot -> dot -> dog -> cog
  dài 6: hit -> hot -> lot -> log -> dog -> cog
  dài 7: hit -> hot -> dot -> lot -> log -> dog -> cog
  dài 7: hit -> hot -> lot -> dot -> dog -> log -> cog
Sau khi lọc ngắn nhất:
  hit -> hot -> dot -> dog -> cog
  hit -> hot -> lot -> log -> cog
Example 2:
  (không có đường nào)


### Nút thắt của Cách 1: số đường đi **bùng nổ**

Với Example 1 chỉ có 7 đỉnh mà đã có 8 đường đơn; chỉ 2 đường là ngắn nhất, 6 đường còn lại là **công cốc**.

Hãy thử một từ điển "đối xứng": mọi từ độ dài `L` chỉ gồm chữ `a`, `b` (ví dụ `L = 3`: `aaa, aab, aba, ...`), đi từ `aa...a` tới `bb...b`. Đồ thị lúc này là một **khối lập phương** `L` chiều; số đường đơn tăng **theo hàm mũ** khi `L` tăng.

In [10]:
for L in (2, 3, 4):
    words = ["".join(t) for t in itertools.product("ab", repeat=L)]
    s = {}
    shortest = find_ladders_dfs_all("a" * L, "b" * L, words, stats=s)
    print(f"L = {L}: {len(words):>2} từ | đường đơn: {len(s['all_paths']):>5} | "
          f"lời gọi dfs: {s['calls']:>6} | đường ngắn nhất: {len(shortest)}")

L = 2:  4 từ | đường đơn:     2 | lời gọi dfs:      5 | đường ngắn nhất: 2
L = 3:  8 từ | đường đơn:    18 | lời gọi dfs:     52 | đường ngắn nhất: 6
L = 4: 16 từ | đường đơn:  6432 | lời gọi dfs:  25013 | đường ngắn nhất: 24


Chỉ với **16 từ** (`L = 4`), DFS đã liệt kê hàng nghìn đường đơn để rồi giữ lại 24 đường. Với `L = 5` (32 từ) con số này còn lớn hơn rất nhiều lần — **đừng chạy thử**. Đề cho tới 500 từ, nên Cách 1 hoàn toàn không dùng được.

Một cải tiến nhỏ: **cắt tỉa** (pruning) — nếu `len(path)` đã vượt độ dài ngắn nhất đã biết thì dừng nhánh. Nhưng DFS có thể "lạc" vào đường dài trước, và số nhánh vẫn có thể là hàm mũ.

**Vấn đề gốc rễ**: DFS đi *sâu* trước, nên nó không biết đường nào là ngắn cho tới khi thử hết. Ta cần một cách duyệt **theo khoảng cách tăng dần**: thăm tất cả các từ cách 1 bước, rồi cách 2 bước, ... Đó chính là **BFS**.

## 8. Nền tảng 5 — Hàng đợi (queue) và `collections.deque`

**Hàng đợi** (queue) hoạt động như xếp hàng mua vé: **vào sau, ra sau** — chính xác hơn là **FIFO** (First In, First Out: ai vào trước ra trước).

- `append(x)`: thêm `x` vào **cuối** hàng.
- `popleft()`: lấy phần tử ở **đầu** hàng ra.

Vì sao không dùng `list` với `pop(0)`? Vì `list.pop(0)` phải **dịch tất cả phần tử còn lại** sang trái một ô → tốn `O(n)`. `deque` (double-ended queue — hàng đợi hai đầu) lấy ở đầu chỉ tốn `O(1)`.

In [11]:
q = deque()
q.append("hit")
q.append("hot")
q.append("dot")
print("hàng đợi:", q)
first = q.popleft()
print("lấy ra:", first, "| còn lại:", q)
q.append("lot")
print("thêm lot:", q, "| len =", len(q))

hàng đợi: deque(['hit', 'hot', 'dot'])
lấy ra: hit | còn lại: deque(['hot', 'dot'])
thêm lot: deque(['hot', 'dot', 'lot']) | len = 3


## 9. Nền tảng 6 — BFS: loang theo từng vòng sóng

**BFS** (Breadth-First Search — tìm kiếm theo chiều rộng) giống như ném hòn đá xuống ao: sóng lan ra **từng vòng**. Vòng 0 là điểm xuất phát, vòng 1 là các đỉnh kề nó, vòng 2 là các đỉnh kề vòng 1 mà chưa thăm, ...

Thuật toán:

1. Cho `start` vào hàng đợi, ghi `dist[start] = 0`.
2. Lặp: lấy `u` ở đầu hàng; với mỗi `v` kề `u` **chưa thăm**: ghi `dist[v] = dist[u] + 1`, cho `v` vào cuối hàng.

Vì hàng đợi FIFO, mọi đỉnh ở vòng `d` đều được lấy ra **trước** mọi đỉnh ở vòng `d + 1`.

In [12]:
def bfs_distances(graph, start):
    dist = {start: 0}
    q = deque([start])
    while q:
        u = q.popleft()
        for v in graph[u]:
            if v not in dist:            # chưa thăm
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

for word, d in bfs_distances(graph_1, "hit").items():
    print(f"dist[{word}] = {d}")

dist[hit] = 0
dist[hot] = 1
dist[dot] = 2
dist[lot] = 2
dist[dog] = 3
dist[log] = 3
dist[cog] = 4


### BFS theo tầng (level by level)

Bài này cần biết rõ **từ nào thuộc tầng nào**, nên ta hay viết BFS theo tầng: giữ cả một list `level` (tầng hiện tại), từ đó sinh `next_level` (tầng kế tiếp). Một kiểu viết tương đương với `deque` là: ở đầu mỗi vòng, `for _ in range(len(q))` để chỉ lấy đúng các phần tử của tầng hiện tại.

In [13]:
def bfs_levels(graph, start):
    levels = [[start]]
    visited = {start}
    while True:
        next_level = []
        for u in levels[-1]:
            for v in graph[u]:
                if v not in visited:
                    visited.add(v)
                    next_level.append(v)
        if not next_level:
            break
        levels.append(next_level)
    return levels

for d, lv in enumerate(bfs_levels(graph_1, "hit")):
    print(f"tầng {d}: {lv}")

tầng 0: ['hit']
tầng 1: ['hot']
tầng 2: ['dot', 'lot']
tầng 3: ['dog', 'log']
tầng 4: ['cog']


### Vì sao BFS cho đường đi ngắn nhất?

Gọi `dist(v)` là số cạnh ít nhất để đi từ `start` tới `v`. BFS giữ bất biến:

> Mọi đỉnh ở tầng `d` có `dist = d` đúng bằng `d`; và tầng `d` được xử lý xong **trước khi** bất kỳ đỉnh tầng `d + 1` nào được xử lý.

Lý do bằng lời: một đỉnh `v` có `dist(v) = d + 1` thì nó kề với ít nhất một đỉnh `u` có `dist(u) = d`. Khi BFS xử lý tầng `d`, nó nhìn thấy `u`, và nếu `v` chưa thăm thì `v` được đưa vào tầng `d + 1`. Không thể nào `v` được phát hiện sớm hơn (vì phát hiện sớm hơn nghĩa là có đường ngắn hơn `d + 1`).

Ở Example 1: `cog` ở tầng 4 → đường ngắn nhất có 4 cạnh = **5 từ**. Khớp với đề!

Nhưng `bfs_distances` mới cho **độ dài**. Đề đòi **liệt kê mọi đường**. Bước tiếp theo: làm sao để BFS "nhớ" đường đi?

## 10. Cách 2 — BFS lưu nguyên đường đi trong hàng đợi

Ý tưởng thẳng thắn: thay vì cho **từ** vào hàng đợi, ta cho **cả đường đi** (một list) vào hàng đợi. Phần tử cuối `path[-1]` là từ đang đứng.

- Lấy một đường `path` ra; với mỗi hàng xóm `nb` của `path[-1]`: tạo đường mới `path + [nb]`.
- Nếu `nb == endWord`: đây là một đường ngắn nhất → đưa vào kết quả.
- BFS theo tầng: ngay khi tầng nào có kết quả thì **xong tầng đó rồi dừng** (các tầng sau chỉ cho đường dài hơn).

**Chi tiết rất quan trọng — "thăm" theo tầng:** một từ có thể được tới từ **nhiều** đường cùng độ dài (ví dụ `cog` tới từ cả `dog` và `log`). Nếu vừa thấy `cog` từ `dog` mà đã đánh dấu "đã thăm" ngay, thì đường qua `log` sẽ bị chặn → mất đáp án! Vì vậy ta gom các từ dùng trong tầng này vào `used_this_level`, và **chỉ sau khi xong cả tầng** mới thêm chúng vào `visited`. (Mục 11 sẽ phân tích kỹ điểm này.)

In [14]:
def find_ladders_bfs_paths(begin_word, end_word, word_list, stats=None):
    if end_word not in word_list:
        return []
    graph = build_graph_pairwise(begin_word, word_list)
    visited = {begin_word}
    queue = deque([[begin_word]])        # mỗi phần tử là MỘT ĐƯỜNG ĐI đầy đủ
    result = []
    stored_paths = 1                     # đếm số đường (bộ phận) từng được lưu
    stored_words = 1                     # tổng số từ nằm trong các đường đó
    while queue and not result:
        used_this_level = set()
        for _ in range(len(queue)):      # chỉ xử lý các đường của tầng hiện tại
            path = queue.popleft()
            for nb in graph[path[-1]]:
                if nb in visited:
                    continue
                new_path = path + [nb]   # SAO CHÉP cả đường: tốn O(độ dài)
                used_this_level.add(nb)
                stored_paths += 1
                stored_words += len(new_path)
                if nb == end_word:
                    result.append(new_path)
                else:
                    queue.append(new_path)
        visited |= used_this_level       # đánh dấu thăm SAU khi xong cả tầng
    if stats is not None:
        stats["stored_paths"] = stored_paths
        stats["stored_words"] = stored_words
    return result

show_paths(find_ladders_bfs_paths(*EXAMPLE_1))
print("Example 2:")
show_paths(find_ladders_bfs_paths(*EXAMPLE_2))

  hit -> hot -> dot -> dog -> cog
  hit -> hot -> lot -> log -> cog
Example 2:
  (không có đường nào)


In [15]:
# Trace Cách 2: hàng đợi chứa gì ở đầu mỗi tầng?
def trace_bfs_paths(begin_word, end_word, word_list):
    graph = build_graph_pairwise(begin_word, word_list)
    visited = {begin_word}
    queue = deque([[begin_word]])
    result, depth = [], 0
    while queue and not result:
        print(f"Tầng {depth}: queue = {list(queue)}")
        used_this_level = set()
        for _ in range(len(queue)):
            path = queue.popleft()
            for nb in graph[path[-1]]:
                if nb in visited:
                    continue
                used_this_level.add(nb)
                if nb == end_word:
                    result.append(path + [nb])
                    print(f"    gặp endWord: {path + [nb]}")
                else:
                    queue.append(path + [nb])
        visited |= used_this_level
        depth += 1
    return result

_ = trace_bfs_paths(*EXAMPLE_1)

Tầng 0: queue = [['hit']]
Tầng 1: queue = [['hit', 'hot']]
Tầng 2: queue = [['hit', 'hot', 'dot'], ['hit', 'hot', 'lot']]
Tầng 3: queue = [['hit', 'hot', 'dot', 'dog'], ['hit', 'hot', 'lot', 'log']]
    gặp endWord: ['hit', 'hot', 'dot', 'dog', 'cog']
    gặp endWord: ['hit', 'hot', 'lot', 'log', 'cog']


Nhìn trace: ở tầng 3, `cog` được tới từ **cả** `dog` và `log` — cả hai đường đều được ghi, nhờ ta chỉ cập nhật `visited` sau khi xong tầng.

### Nút thắt của Cách 2: bộ nhớ

Mỗi phần tử trong hàng đợi là **cả một đường đi**. Hai vấn đề:

1. **Tiền tố bị sao chép lặp lại**: `["hit","hot","dot"]` và `["hit","hot","lot"]` cùng chứa `["hit","hot"]`, nhưng ta lưu 2 lần. Với nhiều đường dài, chi phí là *số đường × độ dài*.
2. **Đường cụt vẫn bị lưu**: BFS không biết trước đường nào sẽ tới được `endWord`, nên nó lưu mọi đường bộ phận — kể cả những đường đi vào "ngõ cụt".

Thử trên một từ điển ngẫu nhiên lớn hơn (80 từ độ dài 3, chữ `a–f`):

In [16]:
random.seed(1)
ALL_3 = ["".join(t) for t in itertools.product("abcdef", repeat=3)]
big_list = random.sample(ALL_3, 80)
big_begin = "aaa"
big_end = max(big_list, key=lambda w: sum(ch != "a" for ch in w))   # một từ "xa" aaa
s = {}
big_res = find_ladders_bfs_paths(big_begin, big_end, big_list, stats=s)
print(f"{big_begin} -> {big_end}: {len(big_res)} đường ngắn nhất, mỗi đường {len(big_res[0])} từ")
print("Số đường bộ phận đã lưu trong hàng đợi:", s["stored_paths"])
print("Tổng số từ đã sao chép vào các đường đó:", s["stored_words"])
show_paths(big_res)

aaa -> feb: 2 đường ngắn nhất, mỗi đường 4 từ
Số đường bộ phận đã lưu trong hàng đợi: 82
Tổng số từ đã sao chép vào các đường đó: 288
  aaa -> aea -> aeb -> feb
  aaa -> aea -> fea -> feb


Chỉ cần vài đường kết quả, nhưng hàng đợi đã chứa hàng chục đường bộ phận (phần lớn là ngõ cụt) và sao chép hàng trăm từ. Trên LeetCode, kiểu lời giải này dễ bị **Time Limit Exceeded / Memory Limit Exceeded** ở các test lớn.

Câu hỏi then chốt: **Có cách nào nhớ đủ thông tin để dựng lại MỌI đường ngắn nhất, mà không lưu từng đường?**

## 11. Insight — Đồ thị cha (`parents`): DAG theo tầng

### Quan sát 1: trên đường ngắn nhất, từ thứ `i` nằm đúng ở tầng `i`

Xét một đường ngắn nhất `w0 = beginWord, w1, w2, ..., wD = endWord`. Khẳng định: `dist(wi) = i` với mọi `i`.

Vì sao? Nếu có `wi` với `dist(wi) < i`, ta có thể đi tới `wi` bằng đường ngắn hơn rồi đi tiếp `wi → ... → wD`, được một đường **ngắn hơn** `D` — mâu thuẫn với "ngắn nhất".

Hệ quả: **mỗi cạnh trên đường ngắn nhất đều đi từ tầng `d` sang tầng `d + 1`**. Các cạnh "ngang" (cùng tầng, như `dot – lot`) hoặc "lùi" (về tầng trước) **không bao giờ** nằm trên đường ngắn nhất.

### Quan sát 2: chỉ cần nhớ "cha" của mỗi từ

Định nghĩa:

> `parents[v]` = danh sách mọi từ `u` ở **tầng ngay trước** `v` mà `u` kề `v`.

Với Example 1:

| Từ `v` | Tầng | `parents[v]` |
|---|---:|---|
| `hot` | 1 | `['hit']` |
| `dot` | 2 | `['hot']` |
| `lot` | 2 | `['hot']` |
| `dog` | 3 | `['dot']` |
| `log` | 3 | `['lot']` |
| `cog` | 4 | `['dog', 'log']` |

Mỗi đường ngắn nhất tương ứng **đúng một cách** đi từ `cog` lần theo `parents` về tới `hit`:

```text
cog ← dog ← dot ← hot ← hit
cog ← log ← lot ← hot ← hit
```

Giống như **gia phả**: thay vì mỗi người cầm một bản gia phả đầy đủ (Cách 2), mỗi người chỉ cần nhớ **bố mẹ mình là ai**; muốn biết dòng họ thì lần ngược lên. Mỗi cạnh tầng `d → d+1` được lưu **đúng một lần**, tiền tố chung không bị sao chép.

Các cạnh `parents` tạo thành một **DAG** (Directed Acyclic Graph — đồ thị có hướng không chu trình), vì cạnh luôn đi từ tầng nhỏ sang tầng lớn hơn nên không thể quay vòng.

### Quan sát 3: vì sao phải xóa từ theo TẦNG, không xóa ngay khi gặp?

Trong BFS thường (mục 9), ta đánh dấu `v` "đã thăm" **ngay khi** gặp lần đầu — vì chỉ cần **một** đường. Ở bài này ta cần **mọi** cha của `v` ở tầng trước.

Xét tầng 3 = `[dog, log]` của Example 1, cả hai đều kề `cog`:

| Cách xóa | Xử lý `dog` | Xử lý `log` | `parents['cog']` |
|---|---|---|---|
| **Xóa ngay** khi gặp | thấy `cog` → ghi cha `dog`, **xóa `cog`** khỏi `unvisited` | `cog` đã bị xóa → **bỏ qua** | `['dog']` ❌ mất đường qua `log` |
| **Xóa theo tầng** | thấy `cog` → ghi cha `dog` (chưa xóa) | thấy `cog` → ghi cha `log` | `['dog', 'log']` ✅ |

Quy tắc đúng:

1. Trong lúc xử lý tầng `d`: `unvisited` vẫn chứa các từ của tầng `d + 1`, nên **mọi** cạnh từ tầng `d` sang chúng đều được ghi.
2. **Sau khi xong cả tầng `d`**: xóa toàn bộ tầng `d + 1` khỏi `unvisited` — từ đây không ai ở tầng `d + 1` hay xa hơn được phép "nhận" chúng làm con nữa (đó sẽ là cạnh ngang/lùi, không nằm trên đường ngắn nhất).

Hàm demo dưới đây cho phép chọn chế độ `"level"` (đúng) hoặc `"immediately"` (sai) để so sánh. Nó dùng `neighbors_az` — cách sinh hàng xóm nhanh mà mục 12 sẽ giảng; tạm thời hiểu là "trả về các từ trong tập kề với `word`".

In [17]:
AZ = "abcdefghijklmnopqrstuvwxyz"

def neighbors_az(word, word_set):
    # Thay từng vị trí bằng từng chữ a-z, giữ lại các từ có trong word_set (giảng kỹ ở mục 12)
    result = []
    for i in range(len(word)):
        for ch in AZ:
            if ch == word[i]:
                continue
            candidate = word[:i] + ch + word[i + 1:]
            if candidate in word_set:
                result.append(candidate)
    return result

def build_parents_demo(begin_word, end_word, word_list, remove="level"):
    unvisited = set(word_list)
    unvisited.discard(begin_word)
    parents = defaultdict(list)
    level, found = [begin_word], False
    while level and not found:
        next_level, seen_next = [], set()
        for word in level:
            for nb in neighbors_az(word, unvisited):
                parents[nb].append(word)
                if remove == "immediately":
                    unvisited.discard(nb)          # SAI: xóa ngay khi gặp
                if nb == end_word:
                    found = True
                if nb not in seen_next:
                    seen_next.add(nb)
                    next_level.append(nb)
        unvisited -= seen_next                     # ĐÚNG: xóa cả tầng sau khi xong tầng
        level = next_level
    return parents, found

for mode in ("immediately", "level"):
    parents, found = build_parents_demo(*EXAMPLE_1, remove=mode)
    print(f"remove={mode:<12} -> parents['cog'] = {parents['cog']}")

remove=immediately  -> parents['cog'] = ['dog']
remove=level        -> parents['cog'] = ['dog', 'log']


### Quan sát 4: vì sao lần ngược từ `endWord`, không đi xuôi từ `beginWord`?

Ta có thể dựng ngược lại "danh sách con" `children[u]` rồi DFS xuôi từ `beginWord`. Nhưng có một khác biệt lớn:

- **Đi ngược** theo `parents` từ `endWord`: mọi cha đều ở tầng ngay trước, nên lần theo cha **luôn** về tới `beginWord` sau đúng `D` bước. **Không có ngõ cụt** — mỗi bước đi đều góp vào ít nhất một đáp án. Công việc tỉ lệ với kích thước đáp án.
- **Đi xuôi** theo `children` từ `beginWord`: BFS đã ghi cạnh tới **mọi** từ ở các tầng, kể cả những từ **không bao giờ dẫn tới `endWord`** (ngõ cụt). DFS xuôi sẽ lạc vào các nhánh đó rồi phải quay lui — tốn công vô ích. (Mục 21 sẽ đo con số cụ thể.)

Tóm lại, lời giải chính gồm 2 pha:

1. **BFS theo tầng** từ `beginWord`, ghi `parents`, xóa theo tầng, dừng sau tầng gặp `endWord`.
2. **Backtrack** (DFS quay lui) từ `endWord` theo `parents` về `beginWord`, đảo ngược đường để in.

Trước khi viết, ta học cách sinh hàng xóm cho nhanh.

## 12. Nền tảng 7 — Ba cách sinh "hàng xóm" của một từ

"Hàng xóm" của `word` = các từ trong từ điển khác `word` đúng một chữ. Có 3 cách tìm:

### (a) So với từng từ trong từ điển

Với mỗi từ, so với `N` từ khác, mỗi lần `O(L)` → mỗi từ tốn `O(N · L)`, cả đồ thị `O(N² · L)`. Đây là cách ở mục 4.

### (b) Thay từng ký tự bằng `a–z`

Từ `hot`: thử mọi vị trí `i` và mọi chữ `ch` khác `word[i]`:

```text
vị trí 0: aot, bot, cot, dot ✅, eot, ..., lot ✅, ..., zot
vị trí 1: hat, hbt, ..., hit, ..., hzt
vị trí 2: hoa, hob, ..., hoz
```

Mỗi ứng viên kiểm tra `in word_set` trung bình `O(1)` (cộng chi phí tạo chuỗi dài `L`). Mỗi từ có `25 · L` ứng viên → `O(25 · L · L)` mỗi từ, **không phụ thuộc `N`**. Với `L <= 5`, đó chỉ là 125 ứng viên mỗi từ!

Cú pháp cần hiểu: `word[:i] + ch + word[i + 1:]` = phần trước vị trí `i` + chữ mới + phần sau vị trí `i`.

In [18]:
word = "hot"
for i in range(len(word)):
    print(f"i={i}: word[:i]='{word[:i]}', word[i]='{word[i]}', word[i+1:]='{word[i + 1:]}' "
          f"-> ví dụ thay bằng 'x': {word[:i] + 'x' + word[i + 1:]}")

word_set_1 = set(EXAMPLE_1[2])
print("neighbors_az('hot') =", neighbors_az("hot", word_set_1))
print("neighbors_az('dog') =", neighbors_az("dog", word_set_1))

i=0: word[:i]='', word[i]='h', word[i+1:]='ot' -> ví dụ thay bằng 'x': xot
i=1: word[:i]='h', word[i]='o', word[i+1:]='t' -> ví dụ thay bằng 'x': hxt
i=2: word[:i]='ho', word[i]='t', word[i+1:]='' -> ví dụ thay bằng 'x': hox
neighbors_az('hot') = ['dot', 'lot']
neighbors_az('dog') = ['cog', 'log', 'dot']


### (c) Wildcard bucket: gom các từ theo "khuôn" `h*t`

Với mỗi từ, tạo `L` **khuôn** (pattern) bằng cách thay một vị trí bằng `*`:

```text
hot -> *ot, h*t, ho*
dot -> *ot, d*t, do*
lot -> *ot, l*t, lo*
```

Hai từ kề nhau ⟺ chúng **chung đúng một khuôn**. Ví dụ `hot`, `dot`, `lot` cùng rơi vào "xô" (bucket) `*ot`.

Ta dựng trước `buckets[pattern] = [các từ khớp khuôn]`. Hàng xóm của `word` = hợp các xô của `L` khuôn của nó, bỏ chính `word`. Dựng xô tốn `O(N · L · L)` (mỗi từ `L` khuôn, mỗi khuôn dài `L`).

In [19]:
def build_buckets(words):
    buckets = defaultdict(list)
    for w in words:
        for i in range(len(w)):
            buckets[w[:i] + "*" + w[i + 1:]].append(w)
    return buckets

def neighbors_bucket(word, buckets):
    result = []
    for i in range(len(word)):
        for other in buckets[word[:i] + "*" + word[i + 1:]]:
            if other != word:
                result.append(other)
    return result

buckets_1 = build_buckets(["hit"] + EXAMPLE_1[2])
for pattern in sorted(buckets_1):
    if len(buckets_1[pattern]) >= 2:          # chỉ in các xô có từ 2 từ trở lên
        print(f"{pattern}: {buckets_1[pattern]}")
print("neighbors_bucket('hot') =", neighbors_bucket("hot", buckets_1))

*og: ['dog', 'log', 'cog']
*ot: ['hot', 'dot', 'lot']
do*: ['dot', 'dog']
h*t: ['hit', 'hot']
lo*: ['lot', 'log']
neighbors_bucket('hot') = ['dot', 'lot', 'hit']


### So sánh chi phí với kích thước tối đa của đề (`N = 500`, `L = 5`)

| Cách | Công việc cho toàn bộ từ điển | Số "đơn vị" công việc |
|---|---|---:|
| (a) So từng cặp | `N(N-1)/2` cặp × `L` ký tự | ≈ 124 750 cặp |
| (b) Thay `a–z` | `N × L × 25` ứng viên | 62 500 ứng viên |
| (c) Wildcard | `N × L` khuôn (dựng xô) + duyệt các xô | 2 500 khuôn |

Cả (b) và (c) đều đủ nhanh. Lời giải chính dùng **(b)** vì ngắn, dễ nhớ, không cần cấu trúc phụ. (c) có lợi khi bảng chữ cái lớn (ví dụ chữ Unicode) hoặc từ điển thưa. Cell dưới kiểm tra 3 cách cho **cùng** tập hàng xóm trên một từ điển ngẫu nhiên.

In [20]:
random.seed(126)
alphabet = "abcde"     # bảng chữ nhỏ để các từ ngẫu nhiên có nhiều hàng xóm
words_500 = sorted({"".join(random.choice(alphabet) for _ in range(5)) for _ in range(800)})[:500]
set_500 = set(words_500)
buckets_500 = build_buckets(words_500)

same = True
for w in words_500:
    a = sorted(x for x in words_500 if differ_by_one(w, x))
    b = sorted(neighbors_az(w, set_500))
    c = sorted(neighbors_bucket(w, buckets_500))
    same = same and (a == b == c)
N, L = len(words_500), 5
print("N =", N, "| 3 cách cho cùng kết quả:", same)
print("So từng cặp :", N * (N - 1) // 2, "cặp")
print("Thay a-z    :", N * L * 25, "ứng viên")
print("Wildcard    :", N * L, "khuôn")

N = 500 | 3 cách cho cùng kết quả: True
So từng cặp : 124750 cặp
Thay a-z    : 62500 ứng viên
Wildcard    : 2500 khuôn


## 13. Cách 3 (cách chính) — BFS theo tầng xây `parents`, rồi backtrack từ `endWord`

### Pha 1: BFS theo tầng xây `parents`

Biến cần có:

- `unvisited`: `set` các từ **chưa được xếp tầng**. Ban đầu là `set(wordList)`, bỏ `beginWord` (nếu `beginWord` có trong list, nó đã ở tầng 0).
- `parents = defaultdict(list)`.
- `level`: list các từ ở tầng hiện tại, bắt đầu `[beginWord]`.
- `found`: đã gặp `endWord` chưa.

Mỗi vòng lặp xử lý **một tầng**:

```text
với mỗi word trong level:
    với mỗi nb trong neighbors_az(word, unvisited):   # nb thuộc tầng kế tiếp
        parents[nb].append(word)                     # ghi MỌI cha
        nếu nb == endWord: found = True              # không dừng ngay! xử lý hết tầng
        thêm nb vào next_level (không trùng)
sau khi xong tầng: unvisited -= next_level            # XÓA THEO TẦNG
level = next_level
```

Điều kiện lặp: `while level and not found` — dừng khi hết từ để loang (không có đường) hoặc **ngay sau tầng** có `endWord`.

Hai chi tiết nhỏ:

- `next_level` cần **không trùng**: `cog` được tới từ cả `dog` và `log` nhưng chỉ được nằm trong tầng sau **một lần** (mục 21 cho thấy trùng sẽ gây đáp án lặp). Ta dùng thêm `seen_next` (set) để kiểm tra nhanh.
- Kiểm tra `endWord in wordList` ngay từ đầu: nếu không có, trả `[]` luôn.

In [21]:
def bfs_build_parents(begin_word, end_word, word_list):
    unvisited = set(word_list)
    if end_word not in unvisited:
        return {}, False
    unvisited.discard(begin_word)
    parents = defaultdict(list)
    level = [begin_word]
    found = False
    while level and not found:
        next_level, seen_next = [], set()
        for word in level:
            for nb in neighbors_az(word, unvisited):
                parents[nb].append(word)          # ghi mọi cha ở tầng trước
                if nb == end_word:
                    found = True                  # vẫn xử lý hết tầng hiện tại
                if nb not in seen_next:
                    seen_next.add(nb)
                    next_level.append(nb)
        unvisited -= seen_next                    # xóa theo TẦNG
        level = next_level
    return parents, found

parents_1, found_1 = bfs_build_parents(*EXAMPLE_1)
print("found =", found_1)
for v in parents_1:
    print(f"parents['{v}'] = {parents_1[v]}")

found = True
parents['hot'] = ['hit']
parents['dot'] = ['hot']
parents['lot'] = ['hot']
parents['dog'] = ['dot']
parents['log'] = ['lot']
parents['cog'] = ['dog', 'log']


Output khớp với bảng ở mục 11. Để ý `parents` **không** có cạnh `dot – lot` hay `dog – log` (cạnh ngang) — chúng bị loại tự động vì khi xử lý `dot`, từ `lot` đã bị xóa khỏi `unvisited` (cuối tầng 1).

### Pha 2: backtrack từ `endWord` về `beginWord`

Dùng đúng khuôn backtracking ở mục 6, nhưng đi **ngược**:

- `path` bắt đầu `[endWord]`.
- Đứng ở `word`: nếu `word == beginWord` → đã đủ một đường; lưu `path[::-1]` (đảo ngược, đồng thời tạo list mới — không bị lỗi tham chiếu).
- Ngược lại: với mỗi `p` trong `parents[word]`: chọn `path.append(p)` → đi tiếp → bỏ chọn `path.pop()`.

Ta không cần `on_path` như Cách 1, vì đồ thị cha là DAG (luôn đi về tầng nhỏ hơn) — không thể đi vòng.

In [22]:
def backtrack_from_end(begin_word, end_word, parents):
    result = []
    path = [end_word]
    def dfs(word):
        if word == begin_word:
            result.append(path[::-1])     # đảo ngược: begin -> ... -> end (list mới)
            return
        for p in parents[word]:
            path.append(p)                # chọn
            dfs(p)                        # đi tiếp
            path.pop()                    # bỏ chọn
    dfs(end_word)
    return result

show_paths(backtrack_from_end("hit", "cog", parents_1))

  hit -> hot -> dot -> dog -> cog
  hit -> hot -> lot -> log -> cog


In [23]:
# Ghép 2 pha thành hàm hoàn chỉnh
def find_ladders_bfs_parents(begin_word, end_word, word_list):
    parents, found = bfs_build_parents(begin_word, end_word, word_list)
    if not found:
        return []
    return backtrack_from_end(begin_word, end_word, parents)

print("Example 1:"); show_paths(find_ladders_bfs_parents(*EXAMPLE_1))
print("Example 2:"); show_paths(find_ladders_bfs_parents(*EXAMPLE_2))
print("beginWord nằm trong wordList, từ dài 1 ký tự:")
show_paths(find_ladders_bfs_parents("a", "c", ["a", "b", "c"]))
print("Thêm 'hog' làm đường tắt:")
show_paths(find_ladders_bfs_parents("hit", "cog", EXAMPLE_1[2] + ["hog"]))

Example 1:
  hit -> hot -> dot -> dog -> cog
  hit -> hot -> lot -> log -> cog
Example 2:
  (không có đường nào)
beginWord nằm trong wordList, từ dài 1 ký tự:
  a -> c
Thêm 'hog' làm đường tắt:
  hit -> hot -> hog -> cog


Ví dụ cuối rất đáng chú ý: thêm `hog` (kề `hot`, `dog`, `log`, `cog`) tạo ra đường tắt `hit -> hot -> hog -> cog` dài 4 từ, nên hai đường dài 5 từ cũ **không còn** là ngắn nhất và biến mất khỏi đáp án.

Còn `"a" -> "c"`: `a` và `c` khác đúng 1 chữ nên kề trực tiếp → đáp án `[["a", "c"]]`, dù `b` cũng kề cả hai.

## 14. Trace từng bước — Example 1 (`hit` → `cog`)

### 14.1 Pha 1: BFS theo tầng

| Tầng | `level` | `unvisited` lúc bắt đầu tầng | Cạnh ghi vào `parents` | Xóa sau tầng |
|---:|---|---|---|---|
| 0 | `hit` | `cog, dog, dot, hot, log, lot` | `hit→hot` | `hot` |
| 1 | `hot` | `cog, dog, dot, log, lot` | `hot→dot`, `hot→lot` | `dot, lot` |
| 2 | `dot, lot` | `cog, dog, log` | `dot→dog`, `lot→log` | `dog, log` |
| 3 | `dog, log` | `cog` | `dog→cog` (**gặp endWord**), `log→cog` (**gặp endWord**) | `cog` |

Ý nghĩa các cột:

- `unvisited` lúc bắt đầu tầng: những từ còn được phép làm "con". Ở tầng 2, `lot` **không** còn trong `unvisited`, nên `dot` không nhận `lot` làm con (cạnh ngang bị bỏ).
- Ở tầng 3, sau khi `dog` gặp `cog`, ta **vẫn** xử lý `log` → `cog` có 2 cha. Hết tầng 3 thì `found = True` nên vòng `while` dừng.

In [24]:
def trace_bfs_parents(begin_word, end_word, word_list):
    unvisited = set(word_list)
    if end_word not in unvisited:
        print("endWord không có trong wordList -> trả về []")
        return {}
    unvisited.discard(begin_word)
    parents = defaultdict(list)
    level, depth, found = [begin_word], 0, False
    while level and not found:
        print(f"Tầng {depth}: level = {level} | unvisited = {sorted(unvisited)}")
        next_level, seen_next = [], set()
        for word in level:
            for nb in neighbors_az(word, unvisited):
                parents[nb].append(word)
                mark = "   <-- GẶP endWord!" if nb == end_word else ""
                print(f"    {word} -> {nb}: parents['{nb}'] = {parents[nb]}{mark}")
                if nb == end_word:
                    found = True
                if nb not in seen_next:
                    seen_next.add(nb)
                    next_level.append(nb)
        unvisited -= seen_next
        print(f"  Hết tầng {depth}: xóa {next_level} khỏi unvisited")
        level, depth = next_level, depth + 1
    print("Dừng: found =", found, "| endWord nằm ở tầng", depth if found else None)
    return parents

trace_parents = trace_bfs_parents(*EXAMPLE_1)

Tầng 0: level = ['hit'] | unvisited = ['cog', 'dog', 'dot', 'hot', 'log', 'lot']
    hit -> hot: parents['hot'] = ['hit']
  Hết tầng 0: xóa ['hot'] khỏi unvisited
Tầng 1: level = ['hot'] | unvisited = ['cog', 'dog', 'dot', 'log', 'lot']
    hot -> dot: parents['dot'] = ['hot']
    hot -> lot: parents['lot'] = ['hot']
  Hết tầng 1: xóa ['dot', 'lot'] khỏi unvisited
Tầng 2: level = ['dot', 'lot'] | unvisited = ['cog', 'dog', 'log']
    dot -> dog: parents['dog'] = ['dot']
    lot -> log: parents['log'] = ['lot']
  Hết tầng 2: xóa ['dog', 'log'] khỏi unvisited
Tầng 3: level = ['dog', 'log'] | unvisited = ['cog']
    dog -> cog: parents['cog'] = ['dog']   <-- GẶP endWord!
    log -> cog: parents['cog'] = ['dog', 'log']   <-- GẶP endWord!
  Hết tầng 3: xóa ['cog'] khỏi unvisited
Dừng: found = True | endWord nằm ở tầng 4


### 14.2 Pha 2: backtrack từ `cog`

Cây lời gọi (mỗi mức thụt vào là một lần đi sang cha):

```text
cog            parents = [dog, log]
├── dog        parents = [dot]
│   └── dot    parents = [hot]
│       └── hot  parents = [hit]
│           └── hit  = beginWord → ghi hit -> hot -> dot -> dog -> cog
└── log        parents = [lot]
    └── lot    parents = [hot]
        └── hot  parents = [hit]
            └── hit  = beginWord → ghi hit -> hot -> lot -> log -> cog
```

Không có nhánh nào "chết": mọi lời gọi đều dẫn tới một đáp án. Đây là lợi ích của việc đi **ngược**.

In [25]:
def trace_backtrack(begin_word, end_word, parents):
    result = []
    path = [end_word]
    def dfs(word, depth):
        indent = "    " * depth
        if word == begin_word:
            found_path = path[::-1]
            print(f"{indent}'{word}' = beginWord -> ghi: {' -> '.join(found_path)}")
            result.append(found_path)
            return
        print(f"{indent}đứng ở '{word}', parents = {parents[word]}, path = {path}")
        for p in parents[word]:
            path.append(p)
            dfs(p, depth + 1)
            path.pop()
    dfs(end_word, 0)
    return result

_ = trace_backtrack("hit", "cog", trace_parents)

đứng ở 'cog', parents = ['dog', 'log'], path = ['cog']
    đứng ở 'dog', parents = ['dot'], path = ['cog', 'dog']
        đứng ở 'dot', parents = ['hot'], path = ['cog', 'dog', 'dot']
            đứng ở 'hot', parents = ['hit'], path = ['cog', 'dog', 'dot', 'hot']
                'hit' = beginWord -> ghi: hit -> hot -> dot -> dog -> cog
    đứng ở 'log', parents = ['lot'], path = ['cog', 'log']
        đứng ở 'lot', parents = ['hot'], path = ['cog', 'log', 'lot']
            đứng ở 'hot', parents = ['hit'], path = ['cog', 'log', 'lot', 'hot']
                'hit' = beginWord -> ghi: hit -> hot -> lot -> log -> cog


## 15. Cách 4 — Cùng thuật toán, đổi cách sinh hàng xóm sang wildcard bucket

Thuật toán **không đổi**; chỉ thay `neighbors_az(word, unvisited)` bằng việc duyệt các xô `buckets[pattern]` và lọc những từ còn trong `unvisited`.

Hai điểm cần để ý:

- Xô phải được dựng từ **tất cả** từ (gồm cả `beginWord`) **một lần** từ đầu; vì xô không tự "quên" từ đã thăm, ta lọc bằng `nb in unvisited`.
- Một hàng xóm `nb` không thể xuất hiện 2 lần cho cùng `word`: hai từ khác đúng 1 vị trí thì chung **đúng một** khuôn. Còn chính `word` có mặt trong cả `L` xô của nó, nhưng nó đã bị xóa khỏi `unvisited` nên tự bị loại.

In [26]:
def find_ladders_buckets(begin_word, end_word, word_list):
    unvisited = set(word_list)
    if end_word not in unvisited:
        return []
    buckets = build_buckets(unvisited | {begin_word})
    unvisited.discard(begin_word)
    parents = defaultdict(list)
    level, found = [begin_word], False
    while level and not found:
        next_level, seen_next = [], set()
        for word in level:
            for i in range(len(word)):
                for nb in buckets[word[:i] + "*" + word[i + 1:]]:
                    if nb not in unvisited:
                        continue
                    parents[nb].append(word)
                    if nb == end_word:
                        found = True
                    if nb not in seen_next:
                        seen_next.add(nb)
                        next_level.append(nb)
        unvisited -= seen_next
        level = next_level
    if not found:
        return []
    return backtrack_from_end(begin_word, end_word, parents)

show_paths(find_ladders_buckets(*EXAMPLE_1))
show_paths(find_ladders_buckets(*EXAMPLE_2))

  hit -> hot -> dot -> dog -> cog
  hit -> hot -> lot -> log -> cog
  (không có đường nào)


## 16. Code nộp LeetCode — cách chính (BFS theo tầng + parents + backtrack từ endWord)

So với bản demo:

- Dùng `set` cho `level` / `next_level` (không cần thứ tự in đẹp nữa, và `set` tự chống trùng).
- Sinh hàng xóm trực tiếp trong vòng lặp; cắt sẵn `prefix`, `suffix` cho mỗi vị trí để đỡ tạo chuỗi lặp lại.
- Không cần bỏ qua `ch == word[i]`: khi đó ứng viên chính là `word`, mà `word` đã bị xóa khỏi `unvisited`.
- Độ sâu đệ quy của backtrack = độ dài đường ≤ `N + 1 = 501`, nằm trong giới hạn đệ quy mặc định của Python (1000).

In [27]:
class Solution:
    def findLadders(self, beginWord: str, endWord: str, wordList: List[str]) -> List[List[str]]:
        unvisited = set(wordList)
        if endWord not in unvisited:
            return []
        unvisited.discard(beginWord)

        # Pha 1: BFS theo tầng, ghi mọi cha ở tầng trước
        parents = defaultdict(list)
        level = {beginWord}
        found = False
        while level and not found:
            next_level = set()
            for word in level:
                for i in range(len(word)):
                    prefix, suffix = word[:i], word[i + 1:]
                    for ch in "abcdefghijklmnopqrstuvwxyz":
                        nb = prefix + ch + suffix
                        if nb in unvisited:
                            parents[nb].append(word)
                            next_level.add(nb)
                            if nb == endWord:
                                found = True
            unvisited -= next_level          # xóa theo TẦNG
            level = next_level

        if not found:
            return []

        # Pha 2: backtrack từ endWord về beginWord
        result = []
        path = [endWord]

        def backtrack(word: str) -> None:
            if word == beginWord:
                result.append(path[::-1])
                return
            for p in parents[word]:
                path.append(p)
                backtrack(p)
                path.pop()

        backtrack(endWord)
        return result

### Bộ test

Các nhóm test:

1. Hai ví dụ của đề.
2. `beginWord` kề trực tiếp `endWord`.
3. `endWord` có trong `wordList` nhưng **không tới được**.
4. `beginWord` **nằm trong** `wordList`.
5. Nhiều đường ngắn nhất "song song" (`aa → ab/ba → bb`).
6. Từ dài 1 ký tự.
7. Có đường tắt làm các đường dài hơn bị loại.

Hàm `check` so sánh sau khi `sorted` (vì đề cho thứ tự tùy ý) và kiểm tra thêm mọi đường đều hợp lệ bằng `is_valid_ladder`.

In [28]:
def check(name, begin_word, end_word, word_list, expected):
    actual = Solution().findLadders(begin_word, end_word, list(word_list))
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    assert all(is_valid_ladder(begin_word, end_word, word_list, p) for p in actual), name
    print(f"{name:<32} PASS ({len(actual)} đường)")

check("Example 1", *EXAMPLE_1, EXPECTED_1)
check("Example 2", *EXAMPLE_2, EXPECTED_2)
check("begin kề end", "hot", "dot", ["dot"], [["hot", "dot"]])
check("end có nhưng không tới được", "hit", "cog", ["hot", "cog"], [])
check("beginWord nằm trong wordList", "hot", "dog", ["hot", "dog", "dot"], [["hot", "dot", "dog"]])
check("2 đường song song", "aa", "bb", ["ab", "ba", "bb"], [["aa", "ab", "bb"], ["aa", "ba", "bb"]])
check("từ dài 1 ký tự", "a", "c", ["a", "b", "c"], [["a", "c"]])
check("đường tắt qua hog", "hit", "cog", EXAMPLE_1[2] + ["hog"], [["hit", "hot", "hog", "cog"]])

Example 1                        PASS (2 đường)
Example 2                        PASS (0 đường)
begin kề end                     PASS (1 đường)
end có nhưng không tới được      PASS (0 đường)
beginWord nằm trong wordList     PASS (1 đường)
2 đường song song                PASS (2 đường)
từ dài 1 ký tự                   PASS (1 đường)
đường tắt qua hog                PASS (1 đường)


### Đối chiếu chéo các cách trên input ngẫu nhiên

Ta sinh nhiều bộ test ngẫu nhiên (cố định `random.seed` để chạy lại ra y hệt):

- **Bộ nhỏ** (≤ 8 từ, độ dài 1–3, chữ `a–c`): đủ nhỏ để chạy cả brute force Cách 1.
- **Bộ vừa** (≤ 60 từ, độ dài 2–4, chữ `a–d`): đối chiếu Cách 2, 3, 4 và `Solution`.

Tất cả phải cho **cùng tập đường** (sau khi `sorted`).

In [29]:
def random_case(rng, max_words, lengths, alphabet):
    L = rng.choice(lengths)
    universe = ["".join(t) for t in itertools.product(alphabet, repeat=L)]
    wl = rng.sample(universe, rng.randint(1, min(max_words, len(universe))))
    begin = rng.choice(universe)
    end = rng.choice(wl)
    return begin, end, wl

rng = random.Random(2026)
small_ok = medium_ok = 0
while small_ok < 300:
    b, e, wl = random_case(rng, 8, [1, 2, 3], "abc")
    if b == e:
        continue
    ref = sorted(find_ladders_dfs_all(b, e, wl))
    assert sorted(find_ladders_bfs_paths(b, e, wl)) == ref, (b, e, wl)
    assert sorted(find_ladders_bfs_parents(b, e, wl)) == ref, (b, e, wl)
    assert sorted(find_ladders_buckets(b, e, wl)) == ref, (b, e, wl)
    assert sorted(Solution().findLadders(b, e, wl)) == ref, (b, e, wl)
    small_ok += 1
while medium_ok < 200:
    b, e, wl = random_case(rng, 60, [2, 3, 4], "abcd")
    if b == e:
        continue
    ref = sorted(find_ladders_bfs_paths(b, e, wl))
    assert sorted(find_ladders_bfs_parents(b, e, wl)) == ref, (b, e, wl)
    assert sorted(find_ladders_buckets(b, e, wl)) == ref, (b, e, wl)
    assert sorted(Solution().findLadders(b, e, wl)) == ref, (b, e, wl)
    medium_ok += 1
print("Bộ nhỏ (có brute force):", small_ok, "test PASS")
print("Bộ vừa                 :", medium_ok, "test PASS")

Bộ nhỏ (có brute force): 300 test PASS
Bộ vừa                 : 200 test PASS


## 17. Vì sao lời giải chính đúng?

Gọi `dist(w)` là số bước ít nhất từ `beginWord` tới `w`, và `D = dist(endWord)`.

**Bất biến 1 (tầng = khoảng cách).** Ở đầu vòng `while` thứ `d` (đếm từ 0):

- `level` chứa **đúng** các từ có `dist = d`;
- `unvisited` chứa **đúng** các từ có `dist > d` (hoặc không tới được).

Đúng khi `d = 0` (`level = [beginWord]`, `unvisited` = mọi từ khác). Giả sử đúng ở vòng `d`: mọi `nb` sinh ra nằm trong `unvisited` và kề một từ có `dist = d`, nên `dist(nb) = d + 1`; ngược lại mọi từ có `dist = d + 1` đều kề ít nhất một từ ở tầng `d` nên đều được sinh ra. Sau `unvisited -= next_level`, bất biến đúng cho vòng `d + 1`.

**Bất biến 2 (đủ cha, không thừa cha).** Sau vòng `d`, với mọi `v` có `dist(v) = d + 1`: `parents[v]` = **mọi** `u` kề `v` có `dist(u) = d`. Lý do "đủ": trong suốt vòng `d`, `v` vẫn còn trong `unvisited` (ta chỉ xóa **sau** khi xong tầng) nên mọi `u` ở tầng `d` kề `v` đều ghi được cạnh. Lý do "không thừa": `u` chỉ ghi cạnh tới từ còn trong `unvisited`, tức tầng `d + 1`.

**Dừng đúng lúc.** Vòng lặp kết thúc ngay sau tầng `D - 1` (khi `found = True`), lúc đó `parents[endWord]` đã đủ mọi cha (Bất biến 2). Nếu `endWord` không tới được, `level` sẽ rỗng và ta trả `[]`.

**Backtrack liệt kê đúng và đủ.** Theo mục 11, mỗi đường ngắn nhất có `wi` ở tầng `i`, nên `w(i-1) ∈ parents[wi]` — tức đường đó là một chuỗi cha từ `endWord` về `beginWord`, và backtrack thử **mọi** chuỗi cha nên không bỏ sót. Ngược lại, mỗi chuỗi cha có `D` bước, từng bước là cạnh hợp lệ → là một đường ngắn nhất. Hai chuỗi cha khác nhau cho hai đường khác nhau → **không trùng**.

## 18. Độ phức tạp

Ký hiệu:

- `N` = số từ trong `wordList` (≤ 500), `L` = độ dài từ (≤ 5).
- `E` = số cạnh được ghi vào `parents` (mỗi từ có tối đa `25 · L` hàng xóm nên `E ≤ 25 · L · N`).
- `P` = tổng số từ trong mọi đường kết quả (đề đảm bảo `P ≤ 10^5`).

**Pha 1 (BFS):** mỗi từ được xử lý trong `level` **tối đa một lần** (vì bị xóa khỏi `unvisited` ngay sau tầng của nó). Mỗi lần sinh `25 · L` (bản LeetCode: `26 · L`) ứng viên, mỗi ứng viên tốn `O(L)` để tạo chuỗi và băm → **`O(N · 26 · L²)`**. Với `N = 500`, `L = 5` là khoảng 325 nghìn phép thao tác ký tự — rất nhanh.

**Pha 2 (backtrack):** vì không có ngõ cụt, mỗi lời gọi đều nằm trên ít nhất một đáp án; số lời gọi ≤ số nút trong "cây đường đi" ≤ `P`. Mỗi lần ghi đáp án tốn `O(D)` để đảo → tổng **`O(P)`**.

**Tổng thời gian:** `O(N · 26 · L² + P)`.

**Bộ nhớ:** `unvisited`, `level`: `O(N · L)`; `parents`: `O(E)`; call stack + `path`: `O(D)`; chưa kể kết quả `O(P)`. → **`O(N · L + E)`** ngoài đầu ra.

So với Cách 2: Cách 2 lưu **mọi đường bộ phận** (kể cả ngõ cụt), có thể lớn theo hàm mũ; Cách 3 chỉ lưu mỗi cạnh một lần.

Cell dưới so sánh bộ nhớ thực tế trên từ điển ngẫu nhiên ở mục 10:

In [30]:
s = {}
_ = find_ladders_bfs_paths(big_begin, big_end, big_list, stats=s)
p_big, _found = bfs_build_parents(big_begin, big_end, big_list)
edges = sum(len(v) for v in p_big.values())
print(f"Cách 2 (BFS lưu đường): {s['stored_paths']} đường bộ phận, {s['stored_words']} từ đã sao chép")
print(f"Cách 3 (parents)      : {len(p_big)} key, {edges} cạnh cha (mỗi cạnh lưu 1 lần)")

Cách 2 (BFS lưu đường): 82 đường bộ phận, 288 từ đã sao chép
Cách 3 (parents)      : 54 key, 72 cạnh cha (mỗi cạnh lưu 1 lần)


## 19. Cách 5 (tùy chọn, nâng cao) — BFS hai chiều (bidirectional BFS)

### Ý tưởng

Thay vì chỉ loang từ `beginWord`, ta loang **đồng thời từ hai đầu**: một "sóng" từ `beginWord`, một "sóng" từ `endWord`, và dừng khi hai sóng **chạm nhau**.

Vì sao nhanh hơn? Giả sử mỗi từ có khoảng `b` hàng xóm và đường ngắn nhất dài `d` bước:

- BFS một chiều thăm khoảng `b^d` từ.
- BFS hai chiều: mỗi sóng chỉ cần đi `d/2` bước → khoảng `2 · b^(d/2)` từ.

Ví dụ `b = 10`, `d = 6`: một chiều ~ `10^6`, hai chiều ~ `2 · 10^3`. Mẹo bổ sung: mỗi vòng luôn **loang phía có tầng nhỏ hơn** để giữ số từ phải xét ít nhất.

### Những điểm phải cẩn thận

1. **Chiều của cạnh:** khi loang từ phía `endWord`, ta đi "ngược". Để dựng lại đường theo chiều `begin → end`, ta lưu `children[u]` = các từ đứng ngay sau `u`. Nếu đang loang xuôi (`forward = True`) và đi `word → nb` thì ghi `children[word].append(nb)`; nếu loang ngược thì ghi `children[nb].append(word)`.
2. **Gặp nhau:** nếu `nb` thuộc tầng hiện tại của **phía bên kia** (`back`), ta đã tìm được đường ngắn nhất → `found = True`, ghi cạnh nối, nhưng **vẫn xử lý hết tầng** (để không mất cạnh nối khác).
3. **Xóa theo tầng** như cũ.
4. Cuối cùng DFS **xuôi** từ `beginWord` theo `children` tới `endWord`. Lưu ý: ở bản này DFS có thể đi vào vài nhánh cụt (các từ ở hai sóng nhưng không nằm trên đường nối) — vẫn đúng, chỉ tốn thêm chút công; muốn triệt để có thể cắt tỉa trước các đỉnh không tới được đích.

In [31]:
def find_ladders_bidirectional(begin_word, end_word, word_list, verbose=False, stats=None):
    word_set = set(word_list)
    if end_word not in word_set:
        return []
    all_words = word_set | {begin_word}           # để phía endWord còn "nhìn thấy" beginWord
    unvisited = word_set - {begin_word, end_word}
    children = defaultdict(list)                  # children[u] = các từ đứng ngay sau u (chiều begin -> end)
    front, back = {begin_word}, {end_word}
    forward, found, expanded = True, False, 0

    def add_edge(word, nb):
        if forward:
            children[word].append(nb)
        else:
            children[nb].append(word)

    while front and back and not found:
        if len(front) > len(back):                # luôn loang phía nhỏ hơn
            front, back = back, front
            forward = not forward
        if verbose:
            print(f"loang {'XUÔI' if forward else 'NGƯỢC'}: front = {sorted(front)} | back = {sorted(back)}")
        next_level = set()
        for word in front:
            expanded += 1
            for nb in neighbors_az(word, all_words):
                if nb in back:                    # hai sóng chạm nhau
                    found = True
                    add_edge(word, nb)
                elif not found and nb in unvisited:
                    next_level.add(nb)
                    add_edge(word, nb)
        unvisited -= next_level
        front = next_level

    if stats is not None:
        stats["expanded"] = expanded
    if not found:
        return []
    result, path = [], [begin_word]
    def dfs(word):
        if word == end_word:
            result.append(path[:])
            return
        for nxt in children[word]:
            path.append(nxt); dfs(nxt); path.pop()
    dfs(begin_word)
    return result

show_paths(find_ladders_bidirectional(*EXAMPLE_1, verbose=True))

loang XUÔI: front = ['hit'] | back = ['cog']
loang XUÔI: front = ['hot'] | back = ['cog']
loang NGƯỢC: front = ['cog'] | back = ['dot', 'lot']
loang NGƯỢC: front = ['dog', 'log'] | back = ['dot', 'lot']
  hit -> hot -> dot -> dog -> cog
  hit -> hot -> lot -> log -> cog


Đọc trace: hai bước đầu loang xuôi (`{hit}` rồi `{hot}`); khi phía xuôi có 2 từ `{dot, lot}` còn phía ngược chỉ có `{cog}`, thuật toán **đổi phía** và loang ngược từ `cog`, được `{dog, log}`. Lần loang tiếp theo, `dog` thấy `dot` và `log` thấy `lot` nằm trong `back` → hai sóng gặp nhau.

### Kiểm tra và đo số từ phải "loang"

Ta đối chiếu với cách chính trên các bộ ngẫu nhiên, rồi so số lần **xử lý một từ** (expand) giữa BFS một chiều và hai chiều trên một từ điển 500 từ (độ dài 4, chữ `a–h`).

In [32]:
rng = random.Random(7)
ok = 0
while ok < 300:
    b, e, wl = random_case(rng, 60, [1, 2, 3, 4], "abcd")
    if b == e:
        continue
    assert sorted(find_ladders_bidirectional(b, e, wl)) == sorted(find_ladders_bfs_parents(b, e, wl)), (b, e, wl)
    ok += 1
print("Bidirectional khớp cách chính trên", ok, "test ngẫu nhiên")

def count_expanded_one_direction(begin_word, end_word, word_list):
    unvisited = set(word_list) - {begin_word}
    level, found, expanded = [begin_word], False, 0
    while level and not found:
        next_level = set()
        for word in level:
            expanded += 1
            for nb in neighbors_az(word, unvisited):
                next_level.add(nb)
                found = found or nb == end_word
        unvisited -= next_level
        level = sorted(next_level)
    return expanded

random.seed(1)
ALL_4 = ["".join(t) for t in itertools.product("abcdefgh", repeat=4)]
list_500 = random.sample(ALL_4, 500)
b500 = list_500[0]
e500 = max(list_500[1:60], key=lambda w: len((find_ladders_bfs_parents(b500, w, list_500) or [[]])[0]))
res500 = find_ladders_bfs_parents(b500, e500, list_500)
s = {}
assert sorted(find_ladders_bidirectional(b500, e500, list_500, stats=s)) == sorted(res500)
print(f"{b500} -> {e500}: {len(res500)} đường ngắn nhất, mỗi đường {len(res500[0])} từ")
print("Số từ phải xử lý — BFS một chiều:", count_expanded_one_direction(b500, e500, list_500))
print("Số từ phải xử lý — BFS hai chiều:", s["expanded"])

Bidirectional khớp cách chính trên 300 test ngẫu nhiên


cbbe -> dgbg: 1 đường ngắn nhất, mỗi đường 10 từ
Số từ phải xử lý — BFS một chiều: 478
Số từ phải xử lý — BFS hai chiều: 41


BFS một chiều gần như phải đi qua cả từ điển, còn BFS hai chiều chỉ xử lý một phần nhỏ. Tuy vậy, với constraints của bài (`N ≤ 500`), cách chính một chiều đã đủ nhanh; BFS hai chiều là "vũ khí" để biết khi gặp đồ thị lớn hơn (ví dụ #127 với từ điển lớn, hoặc đồ thị trạng thái như #752 Open the Lock).

## 20. Bảng so sánh các cách

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Nhận xét |
|---|---|---|---|---|
| 1. DFS liệt kê mọi đường | thử mọi đường đơn, lọc ngắn nhất | hàm mũ (số đường đơn) | `O(N)` + mọi đường | chỉ để hiểu / brute force đối chiếu |
| 2. BFS lưu đường đi | hàng đợi chứa cả đường, visited theo tầng | có thể hàm mũ (số đường bộ phận × độ dài) | rất lớn | đúng nhưng dễ TLE/MLE |
| **3. BFS tầng + parents + backtrack từ end** | lưu mỗi cạnh tầng `d→d+1` một lần, lần ngược từ đích | `O(N·26·L² + P)` | `O(N·L + E)` | **cách chính, nên nộp** |
| 4. Như 3 nhưng wildcard bucket | sinh hàng xóm qua xô `h*t` | worst case như 3, nhưng mỗi từ chỉ tạo `L` khuôn thay vì `25·L` ứng viên | thêm `O(N·L)` cho xô | tốt khi bảng chữ cái lớn / từ điển thưa |
| 5. BFS hai chiều | loang từ hai đầu, luôn loang phía nhỏ | worst case như 3, thực tế ít từ hơn nhiều | như 3 | nâng cao, nhiều chi tiết dễ sai |

(`N` = số từ, `L` = độ dài từ, `E` = số cạnh cha, `P` = tổng kích thước đáp án.)

## 21. Lỗi thường gặp

### Lỗi 1: Xóa từ khỏi `unvisited` ngay khi gặp (giống BFS thường)

```python
for nb in neighbors_az(word, unvisited):
    parents[nb].append(word)
    unvisited.discard(nb)        # SAI: từ cùng tầng phía sau không ghi được cạnh tới nb nữa
```

Hậu quả: mất các cha "đến sau" ở cùng tầng → **mất đáp án**. Sửa: gom tầng sau vào `next_level`, xong cả tầng mới `unvisited -= next_level`.

In [33]:
wrong_parents, _ = build_parents_demo(*EXAMPLE_1, remove="immediately")
print("Xóa ngay  :"); show_paths(backtrack_from_end("hit", "cog", wrong_parents))
right_parents, _ = build_parents_demo(*EXAMPLE_1, remove="level")
print("Xóa theo tầng:"); show_paths(backtrack_from_end("hit", "cog", right_parents))

Xóa ngay  :
  hit -> hot -> dot -> dog -> cog
Xóa theo tầng:
  hit -> hot -> dot -> dog -> cog
  hit -> hot -> lot -> log -> cog


### Lỗi 2: Dừng BFS ngay khi gặp `endWord` giữa tầng

```python
if nb == end_word:
    return ...            # hoặc break ra khỏi mọi vòng lặp — SAI
```

Lúc gặp `cog` từ `dog`, từ `log` (cùng tầng 3) **chưa được xử lý** → cạnh `log → cog` chưa ghi. Sửa: chỉ đặt `found = True`, xử lý **hết tầng** rồi mới dừng (`while level and not found`).

In [34]:
def find_ladders_break_early(begin_word, end_word, word_list):
    unvisited = set(word_list)
    if end_word not in unvisited:
        return []
    unvisited.discard(begin_word)
    parents = defaultdict(list)
    level = [begin_word]
    while level:
        next_level, seen_next, stop = [], set(), False
        for word in level:
            for nb in neighbors_az(word, unvisited):
                parents[nb].append(word)
                if nb == end_word:
                    stop = True                    # SAI: dừng ngay giữa tầng
                    break
                if nb not in seen_next:
                    seen_next.add(nb); next_level.append(nb)
            if stop:
                break
        if stop:
            return backtrack_from_end(begin_word, end_word, parents)
        unvisited -= seen_next
        level = next_level
    return []

print("Dừng sớm (sai):"); show_paths(find_ladders_break_early(*EXAMPLE_1))

Dừng sớm (sai):
  hit -> hot -> dot -> dog -> cog


### Lỗi 3: Tầng sau bị **trùng** từ → đáp án bị lặp

Nếu quên kiểm tra `seen_next` (dùng `list` mà `append` mọi `nb`), một từ có 2 cha sẽ nằm **2 lần** trong tầng sau. Ở tầng kế tiếp, nó được xử lý 2 lần → mỗi con của nó ghi **cùng một cha 2 lần** → backtrack sinh **đường trùng nhau**.

Ví dụ `aaa → bbb` với từ điển gồm mọi từ chữ `a/b`: từ `bba` được tới từ cả `baa` và `aba`. Đúng ra có `3! = 6` đường ngắn nhất.

In [35]:
def find_ladders_duplicate_level(begin_word, end_word, word_list):
    unvisited = set(word_list)
    if end_word not in unvisited:
        return []
    unvisited.discard(begin_word)
    parents, level, found = defaultdict(list), [begin_word], False
    while level and not found:
        next_level = []
        for word in level:
            for nb in neighbors_az(word, unvisited):
                parents[nb].append(word)
                next_level.append(nb)              # SAI: không chống trùng
                found = found or nb == end_word
        unvisited -= set(next_level)
        level = next_level
    return backtrack_from_end(begin_word, end_word, parents) if found else []

cube3 = ["".join(t) for t in itertools.product("ab", repeat=3)]
bad = find_ladders_duplicate_level("aaa", "bbb", cube3)
good = find_ladders_bfs_parents("aaa", "bbb", cube3)
print("Không chống trùng:", len(bad), "đường, trong đó khác nhau:", len(set(map(tuple, bad))))
print("Có chống trùng   :", len(good), "đường")

Không chống trùng: 12 đường, trong đó khác nhau: 6
Có chống trùng   : 6 đường


### Lỗi 4: DFS **xuôi** từ `beginWord` thay vì **ngược** từ `endWord`

Như đã nói ở mục 11, đi xuôi theo `children` sẽ lạc vào các từ không dẫn tới đích. Đo trên từ điển ngẫu nhiên ở mục 10 (đếm số lời gọi DFS):

In [36]:
def count_dfs_calls(begin_word, end_word, parents):
    children = defaultdict(list)
    for v, ps in parents.items():
        for p in ps:
            children[p].append(v)
    forward_calls = backward_calls = 0
    def forward(word):
        nonlocal forward_calls
        forward_calls += 1
        if word == end_word:
            return
        for c in children[word]:
            forward(c)
    def backward(word):
        nonlocal backward_calls
        backward_calls += 1
        if word == begin_word:
            return
        for p in parents[word]:
            backward(p)
    forward(begin_word)
    backward(end_word)
    return forward_calls, backward_calls

f_calls, b_calls = count_dfs_calls(big_begin, big_end, p_big)
print(f"{big_begin} -> {big_end}: số đường ngắn nhất = {len(big_res)}")
print("Số lời gọi DFS xuôi từ beginWord:", f_calls)
print("Số lời gọi DFS ngược từ endWord :", b_calls)

aaa -> feb: số đường ngắn nhất = 2
Số lời gọi DFS xuôi từ beginWord: 82
Số lời gọi DFS ngược từ endWord : 7


DFS xuôi tốn nhiều lần hơn hẳn vì chui vào mọi nhánh cụt; DFS ngược chỉ đi đúng trên các đường đáp án. Trên LeetCode, đây chính là khác biệt giữa **Accepted** và **Time Limit Exceeded** ở các test khó.

### Lỗi 5: Những lỗi nhỏ nhưng hay gặp

| Lỗi | Hậu quả | Sửa |
|---|---|---|
| Quên `unvisited.discard(beginWord)` khi `beginWord` có trong `wordList` | tầng 1 "nhận" `beginWord` làm con → `parents[beginWord]` có giá trị; nếu điều kiện dừng backtrack là "không còn cha" thì sẽ đi vòng `hit → hot → hit → ...` | luôn bỏ `beginWord` khỏi `unvisited` ngay từ đầu, dừng backtrack bằng `word == beginWord` |
| Kiểm tra `nb in wordList` (list) thay vì `set` | mỗi lần kiểm tra `O(N)` → chậm `N` lần | đổi sang `set(wordList)` |
| `result.append(path)` không sao chép | mọi đáp án là cùng một list, cuối cùng bị `pop` rỗng/sai | `path[:]` hoặc `path[::-1]` |
| Quên kiểm tra `endWord` có trong `wordList` | chạy BFS vô ích (vẫn ra `[]` nhưng tốn công) | `if endWord not in unvisited: return []` |
| Tiếp tục BFS sau tầng đã gặp `endWord` | tốn công; nếu code ghi thêm cha cho `endWord` ở tầng sau sẽ ra đường **dài hơn** | điều kiện `while level and not found` |

## 22. Mẹo nhận dạng pattern

- Đề có **"biến đổi từng bước"**, mỗi bước thay đổi nhỏ (một chữ, một chữ số, một vòng khóa...), hỏi **"ít bước nhất"** → nghĩ ngay tới **BFS trên đồ thị trạng thái** (state graph): mỗi trạng thái là đỉnh, mỗi bước là cạnh.
- Đồ thị **không cho sẵn**: các cạnh được **sinh khi cần** (sinh hàng xóm bằng `a–z` hoặc wildcard). Đừng vội dựng cả đồ thị bằng so từng cặp.
- Hỏi **độ dài** ngắn nhất → BFS thường (visited ngay khi gặp). Hỏi **mọi đường** ngắn nhất → BFS theo **tầng** + **ghi mọi cha** + **xóa theo tầng** + **backtrack từ đích**.
- Hỏi **số lượng** đường ngắn nhất (không cần liệt kê) → BFS theo tầng + cộng dồn `ways[nb] += ways[word]` (xem Bài tập 5).
- Từ khóa "all shortest", "every shortest path", "return all sequences" → cảnh giác: đừng lưu cả đường trong hàng đợi.

## 23. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| #127 Word Ladder | cùng đồ thị, chỉ hỏi **độ dài** → BFS thường hoặc BFS hai chiều |
| #433 Minimum Genetic Mutation | giống #127 với bảng chữ `A, C, G, T` |
| #752 Open the Lock | BFS trên trạng thái 4 chữ số, có "ngõ cấm" (deadends) |
| #1091 Shortest Path in Binary Matrix | BFS trên lưới 8 hướng |
| #815 Bus Routes | BFS trên đồ thị "tuyến xe" thay vì "trạm" |
| #1976 Number of Ways to Arrive at Destination | **đếm** số đường ngắn nhất (có trọng số → Dijkstra + đếm) |
| #797 All Paths From Source to Target | liệt kê mọi đường trong DAG bằng backtracking (giống pha 2) |

Ý tưởng "ghi cha để dựng lại đường" (parent pointers) xuất hiện ở hầu hết các thuật toán đường đi: BFS, Dijkstra, A*... Ở bài này ta chỉ mở rộng từ **một** cha thành **danh sách** cha.

## 24. Bài tập vận dụng

Sáu bài, từ dễ đến khó. Các cell TODO chỉ **định nghĩa** hàm; test được để ở dạng comment — làm xong thì bỏ comment để tự kiểm tra. **Đáp án ở cuối notebook** (mục 26).

### Bài 1 — Tính tay

Cho:

```text
beginWord = "red", endWord = "tax"
wordList  = ["ted","tex","red","tax","tad","den","rex","pee"]
```

1. Lập bảng BFS theo tầng như mục 14.1 (cột `level`, cạnh ghi vào `parents`). Để ý `beginWord` nằm trong `wordList`.
2. Viết ra `parents` cuối cùng và **mọi** đường ngắn nhất.
3. Nếu dùng phiên bản **xóa ngay khi gặp** (Lỗi 1), đáp án còn lại bao nhiêu đường? Mất những đường nào?

(Không cần code — dùng giấy bút.)

### Bài 2 — Đoán output

Không chạy, hãy đoán đoạn code sau in ra gì (nhớ thứ tự duyệt của `neighbors_az`: vị trí 0 → L-1, mỗi vị trí chữ `a → z`):

```python
words = {"ted", "tex", "red", "tax", "tad", "den", "rex", "pee"}
print(neighbors_az("tex", words))
b = build_buckets(sorted(words))
print(b["t*x"], b["*ex"], b["te*"], b["p*e"])
```

### Bài 3 — Điền TODO: BFS theo tầng xây `parents`

Hoàn thiện hàm trả về `(parents, found)` giống `bfs_build_parents`. Trọng tâm: **xóa theo tầng**.

In [37]:
def exercise_build_parents(begin_word, end_word, word_list):
    # TODO 1: unvisited = set(word_list); nếu end_word không có -> return {}, False
    # TODO 2: bỏ begin_word khỏi unvisited
    # TODO 3: parents = defaultdict(list); level = [begin_word]; found = False
    # TODO 4: while level and not found:
    #             với mỗi word trong level, mỗi nb trong neighbors_az(word, unvisited):
    #                 ghi parents[nb].append(word); nếu nb == end_word thì found = True
    #                 thêm nb vào next_level nếu chưa có (dùng seen_next)
    # TODO 5: SAU khi xong cả tầng: unvisited -= seen_next; level = next_level
    raise NotImplementedError

# Test gợi ý (bỏ comment sau khi làm xong):
# p, f = exercise_build_parents(*EXAMPLE_1)
# assert f and sorted(p["cog"]) == ["dog", "log"] and "lot" not in p["dot"]
# assert exercise_build_parents(*EXAMPLE_2) == ({}, False)

### Bài 4 — Tự viết: backtrack từ `endWord`

Cho `parents` (kết quả pha 1), viết hàm trả về mọi đường `begin → end`. Không xem lại mục 13!

In [38]:
def exercise_backtrack(begin_word, end_word, parents):
    # TODO: path = [end_word]; DFS theo parents; tới begin_word thì lưu path[::-1]
    # Nhớ khuôn: chọn -> đi tiếp -> bỏ chọn
    pass

# Test gợi ý:
# p, _ = bfs_build_parents(*EXAMPLE_1)
# assert sorted(exercise_backtrack("hit", "cog", p)) == sorted(EXPECTED_1)

### Bài 5 — Biến thể: chỉ **đếm** số đường ngắn nhất

Đôi khi đề chỉ hỏi "có bao nhiêu chuỗi ngắn nhất?". Liệt kê rồi `len(...)` có thể rất chậm (số đường có thể khổng lồ). Hãy đếm **không liệt kê**: trong BFS theo tầng, giữ `ways[w]` = số đường ngắn nhất từ `beginWord` tới `w`.

Gợi ý công thức: `ways[beginWord] = 1`; khi `word` (tầng `d`) kề `nb` (tầng `d + 1`) thì `ways[nb] += ways[word]`.

In [39]:
def exercise_count_shortest_ladders(begin_word, end_word, word_list):
    # TODO: BFS theo tầng giống bài 3, nhưng thay parents bằng ways (dict đếm)
    # Trả về ways[end_word] nếu tới được, ngược lại 0
    pass

# Test gợi ý:
# assert exercise_count_shortest_ladders(*EXAMPLE_1) == 2
# assert exercise_count_shortest_ladders(*EXAMPLE_2) == 0
# assert exercise_count_shortest_ladders("aaaa", "bbbb",
#        ["".join(t) for t in itertools.product("ab", repeat=4)]) == 24

### Bài 6 — Mini challenge: đường ngắn nhất nhỏ nhất theo thứ tự từ điển

Trả về **một** đường ngắn nhất nhỏ nhất theo thứ tự từ điển (so list chuỗi từng phần tử, như `min(...)` của Python), nhưng **không được liệt kê mọi đường** (số đường có thể rất lớn).

Ví dụ Example 1: `["hit","hot","dot","dog","cog"]` (vì `"dot" < "lot"`). Không có đường → `[]`.

Gợi ý hướng: xây `parents`; tìm tập từ **thực sự nằm trên một đường ngắn nhất** (đi ngược từ `endWord`); rồi đi **xuôi** từ `beginWord`, mỗi bước chọn con **nhỏ nhất** còn nằm trên đường ngắn nhất.

In [40]:
def exercise_smallest_ladder(begin_word, end_word, word_list):
    # TODO 1: parents, found = bfs_build_parents(...); nếu not found -> []
    # TODO 2: on_path = các từ đi ngược được từ end_word theo parents (DFS/BFS có visited)
    # TODO 3: children[u] = các v với u trong parents[v] và v trong on_path
    # TODO 4: từ begin_word, mỗi bước chọn min(children[cur]) cho tới end_word
    pass

# Test gợi ý:
# assert exercise_smallest_ladder(*EXAMPLE_1) == ["hit", "hot", "dot", "dog", "cog"]
# assert exercise_smallest_ladder(*EXAMPLE_2) == []

## 25. Gợi ý theo mức

**Bài 1**
- Mức 1: tầng 0 là `[red]`; nhớ bỏ `red` khỏi `unvisited` dù nó có trong `wordList`.
- Mức 2: hàng xóm của `red` trong từ điển: đổi chữ đầu → `ted`; đổi chữ cuối → `rex`.
- Mức 3: tầng 2 có `tex` với **hai** cha; tầng 3 là `tax` cũng với **hai** cha → tổng cộng 3 đường.

**Bài 2**
- Mức 1: `tex` đổi vị trí 0 → `?ex`; vị trí 1 → `t?x`; vị trí 2 → `te?`.
- Mức 2: các xô liệt kê theo thứ tự các từ được đưa vào, ở đây là `sorted(words)`.
- Mức 3: `p*e` chỉ chứa `pee`.

**Bài 3**
- Mức 1: chép khung `while level and not found`.
- Mức 2: `next_level` là list, `seen_next` là set để chống trùng.
- Mức 3: dòng `unvisited -= seen_next` phải nằm **ngoài** vòng `for word in level`.

**Bài 4**
- Mức 1: điểm dừng là `word == begin_word`.
- Mức 2: hàm con lồng bên trong dùng được `path`, `result` của hàm ngoài.
- Mức 3: `for p in parents[word]: path.append(p); dfs(p); path.pop()`.

**Bài 5**
- Mức 1: `ways` là `defaultdict(int)`.
- Mức 2: cộng `ways[nb] += ways[word]` cho **mọi** cặp (giống ghi mọi cha) — vì thế vẫn phải xóa theo tầng.
- Mức 3: vòng ngoài dừng sau tầng có `end_word`; trả `ways[end_word]` nếu `found`.

**Bài 6**
- Mức 1: vì sao không thể chỉ chọn con nhỏ nhất ngay từ `children` đầy đủ? Vì con nhỏ nhất có thể là ngõ cụt.
- Mức 2: `on_path` dùng một stack/`deque` bắt đầu từ `end_word`, thêm `parents[w]` chưa thăm.
- Mức 3: mọi đường ngắn nhất dài bằng nhau, nên so theo thứ tự từ điển = so từng vị trí từ trái sang; chọn tham lam (greedy) nhỏ nhất ở mỗi vị trí là tối ưu, miễn là luôn còn đường tới đích.

## 26. Đáp án tham khảo

Hãy tự làm trước khi xem!

### Đáp án Bài 1 (tính tay)

| Tầng | `level` | Cạnh ghi vào `parents` | Xóa sau tầng |
|---:|---|---|---|
| 0 | `red` | `red→ted`, `red→rex` | `ted, rex` |
| 1 | `ted, rex` | `ted→tad`, `ted→tex`, `rex→tex` | `tad, tex` |
| 2 | `tad, tex` | `tad→tax` (**gặp endWord**), `tex→tax` (**gặp endWord**) | `tax` |

`parents = {ted: [red], rex: [red], tad: [ted], tex: [ted, rex], tax: [tad, tex]}`

Ba đường ngắn nhất (4 từ): `red→ted→tad→tax`, `red→ted→tex→tax`, `red→rex→tex→tax`.

Nếu **xóa ngay**: `tex` bị xóa khi `ted` gặp nó → `rex` không ghi được cạnh tới `tex`; `tax` bị xóa khi `tad` gặp nó → `tex` không ghi được cạnh tới `tax`. Chỉ còn **1** đường `red→ted→tad→tax`. Cell dưới kiểm chứng.

In [41]:
RED_TAX = ("red", "tax", ["ted", "tex", "red", "tax", "tad", "den", "rex", "pee"])
_ = trace_bfs_parents(*RED_TAX)
print("Đáp án đúng:"); show_paths(find_ladders_bfs_parents(*RED_TAX))
wrong_p, _ = build_parents_demo(*RED_TAX, remove="immediately")
print("Nếu xóa ngay:"); show_paths(backtrack_from_end("red", "tax", wrong_p))
assert sorted(find_ladders_bfs_parents(*RED_TAX)) == [["red", "rex", "tex", "tax"],
                                                      ["red", "ted", "tad", "tax"],
                                                      ["red", "ted", "tex", "tax"]]

Tầng 0: level = ['red'] | unvisited = ['den', 'pee', 'rex', 'tad', 'tax', 'ted', 'tex']
    red -> ted: parents['ted'] = ['red']
    red -> rex: parents['rex'] = ['red']
  Hết tầng 0: xóa ['ted', 'rex'] khỏi unvisited
Tầng 1: level = ['ted', 'rex'] | unvisited = ['den', 'pee', 'tad', 'tax', 'tex']
    ted -> tad: parents['tad'] = ['ted']
    ted -> tex: parents['tex'] = ['ted']
    rex -> tex: parents['tex'] = ['ted', 'rex']
  Hết tầng 1: xóa ['tad', 'tex'] khỏi unvisited
Tầng 2: level = ['tad', 'tex'] | unvisited = ['den', 'pee', 'tax']
    tad -> tax: parents['tax'] = ['tad']   <-- GẶP endWord!
    tex -> tax: parents['tax'] = ['tad', 'tex']   <-- GẶP endWord!
  Hết tầng 2: xóa ['tax'] khỏi unvisited
Dừng: found = True | endWord nằm ở tầng 3
Đáp án đúng:
  red -> rex -> tex -> tax
  red -> ted -> tad -> tax
  red -> ted -> tex -> tax
Nếu xóa ngay:
  red -> ted -> tad -> tax


### Đáp án Bài 2

- `neighbors_az("tex", words)`: vị trí 0 → `rex`; vị trí 1 → `tax`; vị trí 2 → `ted` ⇒ `['rex', 'tax', 'ted']`.
- `sorted(words)` = `den, pee, red, rex, tad, tax, ted, tex` nên: `t*x → ['tax', 'tex']`, `*ex → ['rex', 'tex']`, `te* → ['ted', 'tex']`, `p*e → ['pee']`.

In [42]:
words = {"ted", "tex", "red", "tax", "tad", "den", "rex", "pee"}
print(neighbors_az("tex", words))
b = build_buckets(sorted(words))
print(b["t*x"], b["*ex"], b["te*"], b["p*e"])
assert neighbors_az("tex", words) == ["rex", "tax", "ted"]
assert (b["t*x"], b["*ex"], b["te*"], b["p*e"]) == (["tax", "tex"], ["rex", "tex"], ["ted", "tex"], ["pee"])

['rex', 'tax', 'ted']
['tax', 'tex'] ['rex', 'tex'] ['ted', 'tex'] ['pee']


In [43]:
# ===== Đáp án Bài 3, 4, 5, 6 =====
def answer_build_parents(begin_word, end_word, word_list):
    unvisited = set(word_list)
    if end_word not in unvisited:
        return {}, False
    unvisited.discard(begin_word)
    parents = defaultdict(list)
    level, found = [begin_word], False
    while level and not found:
        next_level, seen_next = [], set()
        for word in level:
            for nb in neighbors_az(word, unvisited):
                parents[nb].append(word)
                if nb == end_word:
                    found = True
                if nb not in seen_next:
                    seen_next.add(nb)
                    next_level.append(nb)
        unvisited -= seen_next          # xóa theo tầng
        level = next_level
    return parents, found


def answer_backtrack(begin_word, end_word, parents):
    result, path = [], [end_word]
    def dfs(word):
        if word == begin_word:
            result.append(path[::-1])
            return
        for p in parents[word]:
            path.append(p)
            dfs(p)
            path.pop()
    dfs(end_word)
    return result


def answer_count_shortest_ladders(begin_word, end_word, word_list):
    unvisited = set(word_list)
    if end_word not in unvisited:
        return 0
    unvisited.discard(begin_word)
    ways = defaultdict(int)
    ways[begin_word] = 1
    level, found = [begin_word], False
    while level and not found:
        next_level, seen_next = [], set()
        for word in level:
            for nb in neighbors_az(word, unvisited):
                ways[nb] += ways[word]           # cộng dồn từ MỌI cha
                if nb == end_word:
                    found = True
                if nb not in seen_next:
                    seen_next.add(nb)
                    next_level.append(nb)
        unvisited -= seen_next
        level = next_level
    return ways[end_word] if found else 0


def answer_smallest_ladder(begin_word, end_word, word_list):
    parents, found = answer_build_parents(begin_word, end_word, word_list)
    if not found:
        return []
    on_path, stack = {end_word}, [end_word]      # các từ nằm trên ít nhất 1 đường ngắn nhất
    while stack:
        w = stack.pop()
        for p in parents[w]:
            if p not in on_path:
                on_path.add(p)
                stack.append(p)
    children = defaultdict(list)
    for v in on_path:
        for p in parents[v]:
            children[p].append(v)
    path = [begin_word]
    while path[-1] != end_word:
        path.append(min(children[path[-1]]))     # tham lam: con nhỏ nhất vẫn tới được đích
    return path


p, f = answer_build_parents(*EXAMPLE_1)
assert f and sorted(p["cog"]) == ["dog", "log"] and "lot" not in p["dot"]
assert answer_build_parents(*EXAMPLE_2) == ({}, False)
assert sorted(answer_backtrack("hit", "cog", p)) == sorted(EXPECTED_1)
assert answer_count_shortest_ladders(*EXAMPLE_1) == 2
assert answer_count_shortest_ladders(*EXAMPLE_2) == 0
assert answer_count_shortest_ladders("aaaa", "bbbb", ["".join(t) for t in itertools.product("ab", repeat=4)]) == 24
assert answer_smallest_ladder(*EXAMPLE_1) == ["hit", "hot", "dot", "dog", "cog"]
assert answer_smallest_ladder(*EXAMPLE_2) == []

# Đối chiếu Bài 5 và Bài 6 với lời giải liệt kê trên input ngẫu nhiên
rng = random.Random(99)
cnt = 0
while cnt < 300:
    b_, e_, wl_ = random_case(rng, 40, [2, 3, 4], "abcd")
    if b_ == e_:
        continue
    all_paths = find_ladders_bfs_parents(b_, e_, wl_)
    assert answer_count_shortest_ladders(b_, e_, wl_) == len(all_paths)
    assert answer_smallest_ladder(b_, e_, wl_) == (min(all_paths) if all_paths else [])
    cnt += 1
print("Đáp án Bài 3-6: tất cả assert PASS; đối chiếu ngẫu nhiên", cnt, "test")
print("Ví dụ Bài 6 với red -> tax:", answer_smallest_ladder(*RED_TAX))

Đáp án Bài 3-6: tất cả assert PASS; đối chiếu ngẫu nhiên 300 test
Ví dụ Bài 6 với red -> tax: ['red', 'rex', 'tex', 'tax']


## 27. Template nên nhớ

```python
class Solution:
    def findLadders(self, beginWord, endWord, wordList):
        unvisited = set(wordList)
        if endWord not in unvisited:
            return []
        unvisited.discard(beginWord)
        parents = defaultdict(list)
        level, found = {beginWord}, False
        while level and not found:                      # BFS theo TẦNG
            next_level = set()
            for word in level:
                for i in range(len(word)):
                    for ch in "abcdefghijklmnopqrstuvwxyz":
                        nb = word[:i] + ch + word[i + 1:]
                        if nb in unvisited:
                            parents[nb].append(word)    # ghi MỌI cha
                            next_level.add(nb)
                            found = found or nb == endWord
            unvisited -= next_level                     # xóa theo TẦNG
            level = next_level
        if not found:
            return []
        res, path = [], [endWord]
        def back(w):                                    # lần NGƯỢC từ đích
            if w == beginWord:
                res.append(path[::-1]); return
            for p in parents[w]:
                path.append(p); back(p); path.pop()
        back(endWord)
        return res
```

Câu thần chú: **"Loang theo tầng — ghi mọi cha — xóa cả tầng — xong tầng gặp đích thì dừng — lần ngược từ đích."**

## 28. Checklist trước khi submit

- [ ] `endWord` không có trong `wordList` → trả `[]` ngay.
- [ ] Đã `discard(beginWord)` khỏi `unvisited` (phòng khi `beginWord` có trong `wordList`).
- [ ] Dùng `set` cho `unvisited` (kiểm tra `in` nhanh), không dùng list.
- [ ] Ghi **mọi** cha: `parents[nb].append(word)` cho mọi cặp tầng `d → d+1`.
- [ ] **Xóa theo tầng** (`unvisited -= next_level` sau khi xong tầng), không xóa ngay khi gặp.
- [ ] Gặp `endWord` thì chỉ bật `found`, **xử lý hết tầng** rồi mới dừng; không loang thêm tầng sau.
- [ ] `next_level` không chứa từ trùng (dùng `set`).
- [ ] Backtrack **từ `endWord`** theo `parents`, dừng ở `word == beginWord`, lưu `path[::-1]` (bản sao).
- [ ] Thử tay Example 1 (2 đường), Example 2 (`[]`), và một ca `beginWord` kề `endWord`.